# EnMAP + Landsat 8/9 pairs: final notebook (Member 1, satellite data & preprocessing)

Builds **EnMAP hyperspectral + Landsat 8/9 pairs** for the 40 EnMAP rows of `handover_pairs.csv` (humid, Kerala-like cells) and hands the
passing pairs to Member 2. Same folder pattern and **the same QC v2.3 rules** as the Hyperion and EMIT pairs.
This notebook contains only tested cells. The working notebook `ENMAP_LANDSAT.ipynb` stays as the archive (pilot EN-04 and all outputs).

**Result (24 Oct 2026): 31 of 40 pairs PASS, 23 cells, 177 bands, 30 m, 0.55 GB in `ENMAP-LANDSAT/PASSED-PAIRS/`.**

## Before you start: DLR login
EnMAP files are read directly from DLR's server and need a free DLR account. Put it into **Colab Secrets** (key icon in the left sidebar):
`DLR_USER` and `DLR_PASS`, with *Notebook access* on. The cells read them from there; they are never printed or saved.
Earth Engine (project `hyperspectral-thermal-511303`) is used for the Landsat side.

## Drive layout
```
MEMBER1___PREPROCESSING/ENMAP-LANDSAT/
  config/     enmap_bands.json (wavelengths, FWHM, gain, keep list), enmap_access.json (login method, no credentials)
  raw/        <scene>_METADATA.XML (one example)
  batch/      enmap_items.json (DLR STAC items), enmap_fetch_state.csv (one row per scene: angles, AOD, Landsat used), enmap_band_check.csv
  processed/  <scene>_enmap.tif     cell window: 224 bands (original values) + 6 EnMAP masks + defective count
              <scene>_defective.tif EnMAP defective-pixel mask (224 bands)
              <scene>_landsat.tif   Landsat 8/9 on the EnMAP 30 m grid (_landsat_v1.tif = replaced chip, kept)
  pairs/      <N>/ enmap_reflectance.tif, landsat_<id>.tif, pair_info.json   master archive (PASS and FAIL); pair_index.csv; enmap_final_qc_v2.3.csv
  PASSED-PAIRS/  copies of the PASS pairs for Member 2 + handover_pass.csv + README.txt
```

## Run order
| cell | what | status / when |
|---|---|---|
| **0 STATUS** | overview (read-only) | any time |
| A1 | DLR STAC lookup + first access test | done |
| A2 | DLR login test | done; re-run after changing the account |
| A3 | wavelengths / gain from the metadata, EnMAP masks | done; never overwrites the final band list |
| A4 | fetch all scenes (EnMAP window + masks + Landsat chip) | done (40/40); resumable |
| A5 | band check (spectra, noise, defective share) | done; evidence |
| A6 | final band list (177) + test of the defective-pixel fill | done; writes the keep list |
| **B1** | QC v2.3 -> pair folders | after A4-A6; again after B2 |
| B2 | replace Landsat chips where the screening's scene ends inside the cell | done (7 pairs); `APPLY` is False by default |
| **B3** | handover to PASSED-PAIRS | after every B1 run |

Normal use now: **0 -> B1 -> B3 -> 0**.

## Safety
| cell | writes | can it lose data? |
|---|---|---|
| 0, A5 | nothing | no |
| A1, A2 | STAC cache, `enmap_access.json` | no |
| A3 | `enmap_bands.json` only if it has no keep list yet | no |
| A4 | `processed/` files of scenes not yet fetched | no (resumable, skips finished scenes) |
| A6 | keep list in `enmap_bands.json` | no (same data give the same list) |
| B1 | rewrites `pairs/<N>/` from `processed/` | no (rebuilt from the stored cell files) |
| B2 | only with `APPLY = True`: new Landsat chip; old one kept as `_landsat_v1.tif` | no |
| B3 | `PASSED-PAIRS/`; pairs that no longer pass are **moved** to `_to_delete/` | no |

## 0 STATUS
Read-only overview of every stage.

In [ ]:
# 0. STATUS: where the EnMAP pipeline stands (read-only; changes nothing). Run any time.
import os, glob, json
import pandas as pd

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
PROC, PAIRS, BATCH, CONF, OUT = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch", f"{ROOT}/config", f"{ROOT}/PASSED-PAIRS"

h = pd.read_csv(f"{BASE}/handover_pairs.csv"); en = h[h["sensor"].str.upper() == "ENMAP"]
items = json.load(open(f"{BATCH}/enmap_items.json")) if os.path.exists(f"{BATCH}/enmap_items.json") else {}
print(f"INPUTS: EnMAP rows in handover_pairs.csv: {len(en)} | found in the DLR STAC cache: {sum(1 for v in items.values() if v)}")
cfg = json.load(open(f"{CONF}/enmap_bands.json")) if os.path.exists(f"{CONF}/enmap_bands.json") else {}
acc = json.load(open(f"{CONF}/enmap_access.json")) if os.path.exists(f"{CONF}/enmap_access.json") else {}
print(f"CONFIG: bands {cfg.get('n_bands')} -> keep {len(cfg.get('keep_band_indices', []))} | login method: {acc.get('method')} | window read: {acc.get('window_read')}")

sp = f"{BATCH}/enmap_fetch_state.csv"
st = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame(columns=["scene_id", "fetch_version"])
st2 = st[st["fetch_version"] == 2] if "fetch_version" in st.columns else st.iloc[0:0]
n = {k: len(glob.glob(f"{PROC}/*_{k}.tif")) for k in ("enmap", "defective", "landsat", "landsat_v1")}
rep = int(st2["ls_replaced_from"].notna().sum()) if "ls_replaced_from" in st2.columns else 0
print(f"FETCH: scenes in format v2: {len(st2)} of {len(en)} | files: {n} | Landsat chips replaced (B2): {rep}")

rows = []
for d in glob.glob(f"{PAIRS}/*"):
    if os.path.isdir(d) and os.path.basename(d).isdigit() and os.path.exists(f"{d}/pair_info.json"):
        i = json.load(open(f"{d}/pair_info.json")); rows.append({"pair": int(os.path.basename(d)), "status": i.get("status"),
                                                                 "qc": i.get("qc_version"), "cell": i.get("cell_id")})
pa = pd.DataFrame(rows, columns=["pair", "status", "qc", "cell"])
pp = pa[pa["status"] == "PASS"]
print(f"PAIRS: {len(pa)} | {pa['status'].value_counts().to_dict()} | QC versions: {pa['qc'].value_counts(dropna=False).to_dict()} | PASS cells: {pp['cell'].nunique()}")
copies = {int(os.path.basename(d)) for d in glob.glob(f"{OUT}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()}
want = set(pp["pair"])
print(f"HANDOVER: copies {len(copies)} | PASS not copied {len(want - copies)} | copies no longer PASS {len(copies - want)} -> "
      + ("in sync" if want == copies else "NOT in sync: run B3"))

# PART A: access, bands, fetch
## A1 DLR STAC lookup + first access test
Finds the 40 handover scenes in DLR's STAC collection `ENMAP_HSI_L2A` (cached in `batch/enmap_items.json`) and tries to read one window
without a login. On 24 Oct: 40/40 found; without a login DLR returns its login page ("EnMAP Access Service Login"), so A2 is needed.

In [ ]:
# A1 (= EN-01 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-01: ENMAP-LANDSAT setup + EnMAP rows of the handover list + DLR STAC lookup + access test (one small window). Read-only.
import os, re, json, time
import numpy as np, pandas as pd, requests, rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
for d in ("raw", "processed", "pairs", "batch", "config"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
ITEMS = "https://geoservice.dlr.de/eoc/ogc/stac/v1/collections/ENMAP_HSI_L2A/items"
HALF = 0.035                                          # same 0.07 deg cell as Hyperion / EMIT
os.environ.update({"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.tiff",
                   "GDAL_HTTP_MAX_RETRY": "3", "GDAL_HTTP_RETRY_DELAY": "2"})
pd.set_option("display.width", 220)

# ---- 1) EnMAP rows of the handover list (+ how many more exist in the screening) ----
h = pd.read_csv(f"{BASE}/handover_pairs.csv"); en = h[h["sensor"].str.upper() == "ENMAP"].reset_index(drop=True)
print(f"(1) EnMAP rows in handover_pairs.csv: {len(en)} | unique scenes: {en['scene_id'].nunique()} | cells: {en['cell_id'].nunique()}")
print("    by profile:", en["profile"].value_counts().to_dict())
print("    columns:", list(en.columns))
rp = pd.read_csv(f"{BASE}/screening/ranked_pairs.csv"); rpe = rp[rp["sensor"].str.upper() == "ENMAP"]
print(f"    screening (ranked_pairs.csv): {len(rpe)} EnMAP (scene, cell) rows, {rpe['scene_id'].nunique()} scenes | by profile:",
      rpe.groupby("profile")["scene_id"].nunique().to_dict())

# ---- 2) STAC item for each handover scene (cached in batch/enmap_items.json) ----
cache = f"{ROOT}/batch/enmap_items.json"
items = json.load(open(cache)) if os.path.exists(cache) else {}
def get_item(sid, when, lon, lat):
    try:                                              # direct: /items/{id}
        r = requests.get(f"{ITEMS}/{sid}", params={"f": "application/geo+json"}, timeout=60)
        if r.ok and r.json().get("id") == sid: return r.json()
    except Exception: pass
    t = pd.Timestamp(when)                            # fallback: search by time + point
    r = requests.get(ITEMS, params={"f": "application/geo+json", "limit": 50,
                                    "datetime": f"{(t - pd.Timedelta(minutes=10)).isoformat()}/{(t + pd.Timedelta(minutes=10)).isoformat()}".replace("+00:00", "Z"),
                                    "bbox": f"{lon - 0.01},{lat - 0.01},{lon + 0.01},{lat + 0.01}"}, timeout=120)
    feats = r.json().get("features", []) if r.ok else []
    return next((f for f in feats if f["id"] == sid), feats[0] if feats else None)
for n, r in enumerate(en.drop_duplicates("scene_id").itertuples(), 1):
    if r.scene_id in items: continue
    it = get_item(r.scene_id, r.acq_datetime, float(r.lon), float(r.lat))
    items[r.scene_id] = it
    if n % 10 == 0: print(f"   looked up {n}")
json.dump(items, open(cache, "w"))
found = {k: v for k, v in items.items() if v}
print(f"\n(2) found in the DLR STAC: {len(found)} of {en['scene_id'].nunique()} scenes")
if found:
    ex = next(iter(found.values()))
    print("    asset keys:", list(ex.get("assets", {}).keys()))
    p = ex["properties"]
    print("    example properties:", {k: p.get(k) for k in ("datetime", "eo:cloud_cover", "enmap:overallQuality", "enmap:haze_cover",
                                                         "enmap:cloud_shadow", "version", "proj:epsg")})

# ---- 3) access test: HTTP status of the image + one quality mask, then read one small window ----
if found:
    r0 = en[en["scene_id"] == next(iter(found))].iloc[0]; a = found[r0["scene_id"]]["assets"]
    for key in ("image", "quality_cloud", "metadata"):
        href = (a.get(key) or {}).get("href")
        if not href: print(f"(3) {key}: no href"); continue
        try:
            rr = requests.get(href, headers={"Range": "bytes=0-1023"}, timeout=60, allow_redirects=True)
            print(f"(3) {key}: HTTP {rr.status_code} | final host: {rr.url.split('/')[2]} | auth asked: {rr.headers.get('WWW-Authenticate', 'no')}")
        except Exception as e: print(f"(3) {key}: request failed: {str(e)[:150]}")
    href = a["image"]["href"]
    try:
        t0 = time.time()
        with rasterio.open(href) as src:
            print(f"\n    image: {src.count} bands | {src.width} x {src.height} px | res {src.res} | {src.crs} | dtype {src.dtypes[0]} | nodata {src.nodata}")
            print("    band descriptions (first 3):", src.descriptions[:3], "| tags:", dict(list(src.tags().items())[:8]))
            print("    scales / offsets (first 3):", src.scales[:3], src.offsets[:3])
            b = transform_bounds("EPSG:4326", src.crs, r0["lon"] - HALF, r0["lat"] - HALF, r0["lon"] + HALF, r0["lat"] + HALF)
            win = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
            arr = src.read([1, src.count // 2, src.count], window=win, boundless=True)
        print(f"    cell window {arr.shape[1]} x {arr.shape[2]} px read in {time.time() - t0:.1f} s | values band 1: "
              f"min {np.nanmin(arr[0])}, median {np.nanmedian(arr[0])}, max {np.nanmax(arr[0])}")
        print("\nACCESS OK: EnMAP can be read directly from DLR (no download, no login).")
    except Exception as e:
        print(f"\nREAD FAILED: {str(e)[:300]}\n-> probably needs a DLR login (we handle that in EN-02; do NOT paste credentials here).")

## A2 DLR login test
Tries basic authentication and, if needed, the web-form login; then reads one cell window. On 24 Oct: **basic authentication works**
(HTTP 206, real GeoTIFF), and the window read works. Saves only the method name to `config/enmap_access.json`.

In [ ]:
# A2 (= EN-02 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-02: DLR login test for EnMAP downloads. Credentials from Colab Secrets (DLR_USER / DLR_PASS) or typed into a prompt;
#        never printed, never saved. Saves only the working METHOD name to config/enmap_access.json.
import os, re, json, time, html, getpass
import numpy as np, pandas as pd, requests, rasterio
from urllib.parse import urljoin
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
HALF = 0.035
os.environ.update({"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.tiff",
                   "GDAL_HTTP_MAX_RETRY": "3", "GDAL_HTTP_RETRY_DELAY": "2"})
items = json.load(open(f"{ROOT}/batch/enmap_items.json"))
en = pd.read_csv(f"{BASE}/handover_pairs.csv"); en = en[en["sensor"].str.upper() == "ENMAP"]
sid = next(k for k, v in items.items() if v); a = items[sid]["assets"]
r0 = en[en["scene_id"] == sid].iloc[0]
IMG, QC = a["image"]["href"], a["quality_cloud"]["href"]
is_tiff = lambda b: b[:4] in (b"II*\x00", b"MM\x00*", b"II+\x00", b"MM\x00+")
title = lambda t: (re.search(r"<title>(.*?)</title>", t, re.S | re.I) or [None, "?"])[1].strip()[:80]

# ---- credentials (never printed) ----
try:
    from google.colab import userdata
    USER, PW = userdata.get("DLR_USER"), userdata.get("DLR_PASS")
    print("credentials: from Colab Secrets")
except Exception:
    USER = input("DLR user name: ").strip(); PW = getpass.getpass("DLR password: ")
    print("credentials: typed")

def first_bytes(sess, url, **kw):
    r = sess.get(url, headers={"Range": "bytes=0-1023"}, timeout=90, allow_redirects=True, stream=True, **kw)
    b = r.raw.read(1024, decode_content=True); r.close()
    return r, b

# ---- 1) without login (reference) ----
r, b = first_bytes(requests.Session(), QC)
print(f"\n(1) no login: HTTP {r.status_code}, lands on {r.url.split('/')[2]}, TIFF: {is_tiff(b)}, page title: {title(b.decode('latin1')) if not is_tiff(b) else '-'}")

# ---- 2) method A: basic authentication ----
sA = requests.Session(); sA.auth = (USER, PW)
r, b = first_bytes(sA, QC)
okA = r.ok and is_tiff(b)
print(f"(2) method A (basic auth): HTTP {r.status_code}, lands on {r.url.split('/')[2]}, TIFF: {is_tiff(b)}")

# ---- 3) method B: log in through the SSO web form, keep the session cookies ----
sB, okB = requests.Session(), False
if not okA:
    r = sB.get(QC, timeout=90, allow_redirects=True)
    m = re.search(r'<form[^>]*action="([^"]+)"', r.text, re.I)
    if not m: print("(3) method B: no login form found on", r.url.split("/")[2], "| title:", title(r.text))
    else:
        data = dict(re.findall(r'<input[^>]*type="hidden"[^>]*name="([^"]+)"[^>]*value="([^"]*)"', r.text, re.I))
        names = re.findall(r'<input[^>]*name="([^"]+)"', r.text, re.I)
        ufield = next((n for n in names if n.lower() in ("username", "user", "email", "login")), "username")
        pfield = next((n for n in names if "pass" in n.lower()), "password")
        data.update({ufield: USER, pfield: PW})
        r2 = sB.post(urljoin(r.url, html.unescape(m.group(1))), data=data, timeout=90, allow_redirects=True)
        if "invalid" in r2.text.lower() and "password" in r2.text.lower(): print("(3) method B: login page says the user name or password is wrong")
        r3, b3 = first_bytes(sB, QC)
        okB = r3.ok and is_tiff(b3)
        print(f"(3) method B (web login): HTTP {r3.status_code}, lands on {r3.url.split('/')[2]}, TIFF: {is_tiff(b3)}"
              + ("" if okB else f", page title: {title(b3.decode('latin1'))}"))

# ---- 4) can rasterio read just the cell window? (the efficient way) ----
method = "basic" if okA else "sso_cookie" if okB else None
win_ok = False
if method:
    if method == "basic":
        os.environ.update({"GDAL_HTTP_AUTH": "BASIC", "GDAL_HTTP_USERPWD": f"{USER}:{PW}"})
    else:
        os.environ["GDAL_HTTP_COOKIE"] = "; ".join(f"{c.name}={c.value}" for c in sB.cookies)
    try:
        t0 = time.time()
        with rasterio.open(IMG) as src:
            b = transform_bounds("EPSG:4326", src.crs, r0["lon"] - HALF, r0["lat"] - HALF, r0["lon"] + HALF, r0["lat"] + HALF)
            win = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
            arr = src.read([1, src.count // 2, src.count], window=win, boundless=True)
            print(f"\n(4) image: {src.count} bands | {src.width} x {src.height} px | res {src.res} | {src.crs} | dtype {src.dtypes[0]} | nodata {src.nodata}")
            print("    band descriptions (first 3):", src.descriptions[:3], "| scales/offsets:", src.scales[:2], src.offsets[:2])
            print("    tags:", dict(list(src.tags().items())[:8]))
        print(f"    cell window {arr.shape[1]} x {arr.shape[2]} px, 3 bands read in {time.time() - t0:.1f} s | band 1: "
              f"min {np.nanmin(arr[0])}, median {np.nanmedian(arr[0])}, max {np.nanmax(arr[0])}")
        win_ok = True
    except Exception as e:
        print(f"\n(4) window read failed: {str(e)[:250]}")
    if not win_ok:                                                   # fallback info: how big is a whole file?
        s = sA if method == "basic" else sB
        hd = s.head(IMG, allow_redirects=True, timeout=90)
        print(f"    fallback = download whole files: image size {int(hd.headers.get('Content-Length', 0)) / 1e6:.0f} MB per scene")

json.dump({"method": method, "window_read": win_ok, "tested_scene": sid, "tested": time.strftime("%Y-%m-%d %H:%M")},
          open(f"{ROOT}/config/enmap_access.json", "w"), indent=1)
print(f"\nRESULT: login method = {method} | window read = {win_ok}  (saved to config/enmap_access.json, no credentials stored)")
if not method: print("Neither method worked: check the account in a browser first (log in, open an EnMAP link, accept any licence).")

## A3 Wavelengths, gain and EnMAP masks
Reads the band characterisation from the scene's METADATA.XML: **224 bands, 418.4-2445.3 nm, gain 0.0001, offset 0** (reflectance =
value x 0.0001). DLR leaves out 1390-1449 nm and 1780-1968 nm. Lists EnMAP's own masks over the cell (cloud, cloud shadow, haze, cirrus,
snow, land/water classes, defective pixels). Changed from EN-03: it never overwrites a band config that already holds the final keep list,
and the speed test is optional (`SPEED_TEST`).

In [ ]:
# A3 (= EN-03 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-03: EnMAP band details (wavelengths / FWHM / gain from the metadata XML) + quality masks + speed test
#        (A: window read of many bands in one call, B: whole-file download to Colab's temporary disk). Credentials never printed.
import os, re, json, time
import numpy as np, pandas as pd, requests, rasterio
import xml.etree.ElementTree as ET
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
TMP = "/content/enmap_tmp"; os.makedirs(TMP, exist_ok=True)             # Colab local disk, temporary (not Drive)
HALF = 0.035
SPEED_TEST = False                     # True = repeat the speed test (downloads one 466 MB file to Colab's temporary disk)
try:
    from google.colab import userdata; USER, PW = userdata.get("DLR_USER"), userdata.get("DLR_PASS")
except Exception:
    import getpass; USER = input("DLR user name: ").strip(); PW = getpass.getpass("DLR password: ")
os.environ.update({"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.tiff",
                   "GDAL_HTTP_MAX_RETRY": "3", "GDAL_HTTP_RETRY_DELAY": "2", "GDAL_HTTP_AUTH": "BASIC", "GDAL_HTTP_USERPWD": f"{USER}:{PW}",
                   "GDAL_HTTP_MULTIRANGE": "YES", "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES", "GDAL_HTTP_VERSION": "2",
                   "VSI_CACHE": "TRUE", "VSI_CACHE_SIZE": str(512 * 1024 * 1024), "CPL_VSIL_CURL_CHUNK_SIZE": str(4 * 1024 * 1024)})
S = requests.Session(); S.auth = (USER, PW)
items = json.load(open(f"{ROOT}/batch/enmap_items.json"))
en = pd.read_csv(f"{BASE}/handover_pairs.csv"); en = en[en["sensor"].str.upper() == "ENMAP"]
sid = next(k for k, v in items.items() if v); a = items[sid]["assets"]; r0 = en[en["scene_id"] == sid].iloc[0]
tag = lambda e: e.tag.split("}")[-1]
def cell_window(src):
    b = transform_bounds("EPSG:4326", src.crs, r0["lon"] - HALF, r0["lat"] - HALF, r0["lon"] + HALF, r0["lat"] + HALF)
    return from_bounds(*b, transform=src.transform).round_offsets().round_lengths()

# ---- 1) metadata XML: wavelengths, FWHM, gain / offset per band (FIXED: only entries that carry wavelengthCenterOfBand) ----
xp = f"{ROOT}/raw/{sid}_METADATA.XML"
if os.path.exists(xp): content = open(xp, "rb").read()
else:
    x = S.get(a["metadata"]["href"], timeout=120); x.raise_for_status(); content = x.content; open(xp, "wb").write(content)
root = ET.fromstring(content)
spec = {}
for el in root.iter():
    if tag(el) == "bandID":
        d = {tag(c): (c.text or "").strip() for c in el}
        if d.get("wavelengthCenterOfBand") and el.attrib.get("number"): spec.setdefault(int(el.attrib["number"]), d)
nums = sorted(spec)
f = lambda d, k: float(d[k]) if d.get(k) not in (None, "") else None
cfg = {"n_bands": len(nums), "band_number": nums,
       "wavelength_nm": [f(spec[n], "wavelengthCenterOfBand") for n in nums], "fwhm_nm": [f(spec[n], "FWHMOfBand") for n in nums],
       "gain": [f(spec[n], "GainOfBand") for n in nums], "offset": [f(spec[n], "OffsetOfBand") for n in nums],
       "source": f"METADATA.XML of {sid} (bandID entries with wavelengthCenterOfBand)"}
wl = np.array(cfg["wavelength_nm"], float)
print(f"(1) bands with wavelength: {cfg['n_bands']} (numbers {nums[0]}-{nums[-1]}) | {wl.min():.1f}-{wl.max():.1f} nm"
      f" | gain values: {sorted(set(cfg['gain']))} | offset values: {sorted(set(cfg['offset']))}")
dwl = np.diff(wl); print(f"    band spacing: median {np.median(dwl):.1f} nm | gaps > 15 nm at:",
                         [(round(wl[i]), round(wl[i + 1])) for i in np.where(dwl > 15)[0]] or "none",
                         "| non-increasing steps (VNIR/SWIR overlap):", [(round(wl[i]), round(wl[i + 1])) for i in np.where(dwl <= 0)[0]] or "none")
cfgp = f"{ROOT}/config/enmap_bands.json"
old = json.load(open(cfgp)) if os.path.exists(cfgp) else {}
if "keep_band_indices" in old:                                          # SAFETY: keep the band list made in A6
    same = np.allclose(old["wavelength_nm"], cfg["wavelength_nm"])
    print("    enmap_bands.json already has the final band list -> NOT overwritten | wavelengths identical:", same)
else: json.dump(cfg, open(cfgp, "w"))

# ---- 2) EnMAP's own quality masks over the cell ----
print("\n(2) quality masks over the cell (value: share of pixels)")
for k in ("quality_cloud", "quality_cloud_shadow", "quality_haze", "quality_cirrus", "quality_snow", "quality_classes", "defective_pixel_mask"):
    href = (a.get(k) or {}).get("href")
    if not href: print(f"    {k}: missing"); continue
    try:
        t0 = time.time()
        with rasterio.open(href) as src:
            m = src.read(1, window=cell_window(src), boundless=True); info = f"{src.count} band(s), {src.dtypes[0]}"
        v, c = np.unique(m, return_counts=True)
        print(f"    {k}: {info} | {dict(zip(v.tolist()[:8], np.round(c / c.sum(), 3).tolist()[:8]))} | {time.time() - t0:.1f} s")
    except Exception as e: print(f"    {k}: read failed: {str(e)[:120]}")

if SPEED_TEST:
    # ---- 3) speed test A: many bands in ONE window read ----
    NB = 32
    with rasterio.open(a["image"]["href"]) as src:
        print(f"\n(3) image layout: interleave {src.interleaving}, blocks {src.block_shapes[0]}, compression {src.compression}, overviews {src.overviews(1)}")
        t0 = time.time(); arr = src.read(list(range(1, NB + 1)), window=cell_window(src), boundless=True); tA = time.time() - t0
    print(f"    A: {NB} bands in one call: {tA:.1f} s -> all 224 bands about {tA * 224 / NB / 60:.1f} min per scene")

    # ---- 4) speed test B: download the whole file, cut the cell locally, delete the temporary copy ----
    fp = f"{TMP}/{sid}_image.tif"; t0 = time.time()
    with S.get(a["image"]["href"], stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(fp, "wb") as o:
            for ch in r.iter_content(8 * 1024 * 1024): o.write(ch)
    tdl, size = time.time() - t0, os.path.getsize(fp) / 1e6
    t0 = time.time()
    with rasterio.open(fp) as src: full = src.read(window=cell_window(src), boundless=True)
    tB = tdl + time.time() - t0
    os.remove(fp)                                                            # temporary file on Colab's disk only
    print(f"    B: whole file {size:.0f} MB downloaded in {tdl:.0f} s ({size / max(tdl, 1e-6):.1f} MB/s) + cell cut -> {tB / 60:.1f} min per scene"
          f" | cell {full.shape} | band 1 median {np.median(full[0][full[0] != -32768]):.0f}")

    best = "window" if tA * 224 / NB < tB else "download"
    acc = json.load(open(f"{ROOT}/config/enmap_access.json")); acc.update({"read_method": best, "minutes_per_scene_A": round(tA * 224 / NB / 60, 1),
                                                                         "minutes_per_scene_B": round(tB / 60, 1)})
    json.dump(acc, open(f"{ROOT}/config/enmap_access.json", "w"), indent=1)
    print(f"\nRESULT: faster method = {best} (A ~{tA * 224 / NB / 60:.1f} min, B ~{tB / 60:.1f} min per scene) | for 40 scenes: about "
          f"{40 * min(tA * 224 / NB, tB) / 60:.0f} min")
else: print("\n(3)-(4) speed test skipped (SPEED_TEST = False). Result on 24 Oct: window read ~1 min per scene (used).")

## A4 Fetch all scenes
For each scene: the EnMAP cell window (all 224 bands in one read, original values), the 6 EnMAP masks, the defective-pixel mask, and
Landsat 8/9 surface reflectance, LST and clear fraction **on exactly the EnMAP 30 m grid** (Earth Engine `computePixels`, bilinear).
Also records sun and viewing angles, AOD and water vapour from the STAC item. About 1 min per scene (40 scenes in 36 min). Resumable.
It also checks whether DLR repaired the defective pixels: on 24 Oct **no** (62 % are nodata in the source, the rest differ from their
neighbours by 30 % against 5 % for normal pixels).

In [ ]:
# A4 (= EN-06 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-06: FETCH ALL EnMAP scenes (format v2): EnMAP cell window with ORIGINAL values (224 bands) + 6 EnMAP masks + defective count,
#        full defective mask (separate small file), Landsat 8/9 on the same 30 m grid, scene angles / AOT / WV.
#        + check: are 'defective' pixels repaired by DLR (look like their neighbours) or garbage? Resumable (~1 min / scene).
import os, re, json, time
import numpy as np, pandas as pd, rasterio, ee
from rasterio.windows import from_bounds, Window
from rasterio.warp import transform_bounds

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
PROC, BATCH = f"{ROOT}/processed", f"{ROOT}/batch"
try: ee.Initialize(project="hyperspectral-thermal-511303")
except Exception: ee.Authenticate(); ee.Initialize(project="hyperspectral-thermal-511303")
try:
    from google.colab import userdata; USER, PW = userdata.get("DLR_USER"), userdata.get("DLR_PASS")
except Exception:
    import getpass; USER = input("DLR user name: ").strip(); PW = getpass.getpass("DLR password: ")
os.environ.update({"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.tiff",
                   "GDAL_HTTP_MAX_RETRY": "3", "GDAL_HTTP_RETRY_DELAY": "2", "GDAL_HTTP_AUTH": "BASIC", "GDAL_HTTP_USERPWD": f"{USER}:{PW}",
                   "GDAL_HTTP_MULTIRANGE": "YES", "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES", "GDAL_HTTP_VERSION": "2",
                   "VSI_CACHE": "TRUE", "VSI_CACHE_SIZE": str(1024 * 1024 * 1024), "CPL_VSIL_CURL_CHUNK_SIZE": str(4 * 1024 * 1024)})

FETCH_VERSION = 2                                                   # v2 = original values kept; defective mask stored separately
HALF = 0.035
MASKS = ["quality_cloud", "quality_cloud_shadow", "quality_haze", "quality_cirrus", "quality_snow", "quality_classes"]
LSBANDS = ["blue", "green", "red", "nir", "swir1", "swir2", "LST_K", "clear_frac"]
PROPS = {"sun_elev": "view:sun_elevation", "sun_az": "view:sun_azimuth", "across_offnadir": "enmap:acrossOffNadirAngle",
         "along_offnadir": "enmap:alongOffNadirAngle", "scene_az": "enmap:sceneAzimuthAngle", "aot": "enmap:sceneAOT", "wv": "enmap:sceneWV",
         "cloud_cover": "eo:cloud_cover", "haze_cover": "enmap:haze_cover", "shadow_cover": "enmap:cloud_shadow", "version": "version"}
WL = np.array(json.load(open(f"{ROOT}/config/enmap_bands.json"))["wavelength_nm"])
items = json.load(open(f"{BATCH}/enmap_items.json"))
en = pd.read_csv(f"{BASE}/handover_pairs.csv"); en = en[en["sensor"].str.upper() == "ENMAP"].drop_duplicates("scene_id").reset_index(drop=True)

def cell_window(src, lon, lat):
    b = transform_bounds("EPSG:4326", src.crs, lon - HALF, lat - HALF, lon + HALF, lat + HALF)
    return from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
def read_window(src, win, fill, indexes=None):
    n = src.count if indexes is None else len(indexes)
    out = np.full((n, int(win.height), int(win.width)), fill, dtype=src.dtypes[0])
    try: inter = win.intersection(Window(0, 0, src.width, src.height))
    except Exception: return out
    data = src.read(indexes, window=inter)
    r0, c0 = int(inter.row_off - win.row_off), int(inter.col_off - win.col_off)
    out[:, r0:r0 + data.shape[1], c0:c0 + data.shape[2]] = data
    return out
def landsat_chip(lid, epsg, tf, w, h):
    coll = "LANDSAT/LC09/C02/T1_L2" if lid.startswith("LC09") else "LANDSAT/LC08/C02/T1_L2"
    L = ee.Image(f"{coll}/{lid}"); qa = L.select("QA_PIXEL")
    clear = qa.bitwiseAnd(0b11110).eq(0).And(qa.neq(0))
    sr = L.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]).multiply(0.0000275).add(-0.2).rename(LSBANDS[:6])
    lst = L.select("ST_B10").multiply(0.00341802).add(149.0).rename("LST_K")
    img = (sr.addBands(lst).updateMask(clear).resample("bilinear")
             .addBands(clear.rename("clear_frac").toFloat().resample("bilinear")).toFloat().unmask(-9999))
    grid = {"dimensions": {"width": w, "height": h}, "crsCode": f"EPSG:{epsg}",
            "affineTransform": {"scaleX": tf.a, "shearX": tf.b, "translateX": tf.c, "shearY": tf.d, "scaleY": tf.e, "translateY": tf.f}}
    arr = ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY", "grid": grid})
    A = np.stack([arr[b] for b in LSBANDS]).astype("float32"); A[A == -9999] = np.nan
    return A
def defect_check(E, D):                     # do defective pixels look like their left/right neighbours (= repaired)?
    Ef = E.astype("float32"); Ef[E == -32768] = np.nan
    c, nb = Ef[:, :, 1:-1], (Ef[:, :, :-2] + Ef[:, :, 2:]) / 2
    rel = np.abs(c - nb) / np.maximum(np.abs(nb), 1.0)
    okn = (D[:, :, :-2] == 0) & (D[:, :, 2:] == 0) & np.isfinite(rel)
    d = (D[:, :, 1:-1] == 1) & okn; g = (D[:, :, 1:-1] == 0) & okn
    dn = float((E[D == 1] == -32768).mean()) if (D == 1).any() else np.nan
    return (float(np.median(rel[d])) if d.any() else np.nan), float(np.median(rel[g][::50])), dn

# ---- FIXED: the pilot rows from EN-04 have no fetch_version -> mark them as version 1 (they are refetched in v2) ----
sp = f"{BATCH}/enmap_fetch_state.csv"
state = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame(columns=["scene_id", "fetch_version"])
if "fetch_version" not in state.columns: state["fetch_version"] = 1
done = set(state.loc[state["fetch_version"] == FETCH_VERSION, "scene_id"])
print(f"scenes: {len(en)} | already fetched in format v{FETCH_VERSION}: {len(done)}")
t_all, n_new = time.time(), 0
for n, r in en.iterrows():
    sid = r["scene_id"]; it = items.get(sid)
    fe, fd, fl = f"{PROC}/{sid}_enmap.tif", f"{PROC}/{sid}_defective.tif", f"{PROC}/{sid}_landsat.tif"
    if sid in done and all(os.path.exists(x) for x in (fe, fd, fl)): continue
    if not it: print("no STAC item:", sid[:60]); continue
    a, p = it["assets"], it["properties"]; lid = re.sub(r"^\d+_", "", str(r["ls_ids"])); t0 = time.time()
    try:
        with rasterio.open(a["image"]["href"]) as src:
            win = cell_window(src, float(r["lon"]), float(r["lat"])); tf, crs, epsg = src.window_transform(win), src.crs, src.crs.to_epsg()
            E = read_window(src, win, -32768)
        with rasterio.open(a["defective_pixel_mask"]["href"]) as src: D = read_window(src, win, 0)
        M = []
        for k in MASKS:
            with rasterio.open(a[k]["href"]) as src: M.append(read_window(src, win, 255, [1])[0])
        M = np.stack(M); L = landsat_chip(lid, epsg, tf, int(win.width), int(win.height))
    except Exception as e:
        print(f"FAILED {sid[:50]}: {str(e)[:200]}"); continue
    nb = E.shape[0]; dcount = (D == 1).sum(axis=0).astype("int16")
    with rasterio.open(fe, "w", driver="GTiff", height=E.shape[1], width=E.shape[2], count=nb + len(MASKS) + 1, dtype="int16",
                       crs=crs, transform=tf, nodata=-32768, compress="deflate", tiled=True) as dst:
        dst.write(E, list(range(1, nb + 1))); dst.write(M.astype("int16"), list(range(nb + 1, nb + len(MASKS) + 1)))
        dst.write(dcount, nb + len(MASKS) + 1)
        for k in range(nb): dst.set_band_description(k + 1, f"{WL[k]:.1f} nm (EnMAP b{k + 1:03d})")
        for j, k in enumerate(MASKS): dst.set_band_description(nb + 1 + j, k)
        dst.set_band_description(nb + len(MASKS) + 1, "defective_band_count")
        dst.update_tags(scale_factor=0.0001, units="surface reflectance x 10000 (bands 1-224, ORIGINAL values); masks 0/1, 255 = outside the tile",
                        source=sid, product="EnMAP L2A (DLR)", proc_version=str(p.get("version")), fetch_version=FETCH_VERSION)
    with rasterio.open(fd, "w", driver="GTiff", height=D.shape[1], width=D.shape[2], count=nb, dtype="uint8",
                       crs=crs, transform=tf, compress="deflate", tiled=True) as dst:
        dst.write(D.astype("uint8")); dst.update_tags(note="EnMAP defective_pixel_mask for the cell, one band per spectral band (1 = defective)")
    with rasterio.open(fl, "w", driver="GTiff", height=L.shape[1], width=L.shape[2], count=len(LSBANDS), dtype="float32",
                       crs=crs, transform=tf, nodata=np.nan, compress="deflate") as dst:
        dst.write(L)
        for k, nme in enumerate(LSBANDS, start=1): dst.set_band_description(k, nme)
        dst.update_tags(landsat_scene=lid, note="Landsat bilinear-resampled onto the EnMAP 30 m grid; clear_frac = clear share; NaN = no data")
    rel_def, rel_good, def_nodata = defect_check(E, D)
    inside = M[0] != 255
    row = {"scene_id": sid, "fetch_version": FETCH_VERSION, "date": str(r["acq_datetime"])[:10], "cell_id": r["cell_id"],
           "lon": r["lon"], "lat": r["lat"], "ls_id": lid, "ls_dt": r["ls_dt"], "kg": r["kg"], "profile": r["profile"],
           "epsg": epsg, "w": int(win.width), "h": int(win.height), **{k: p.get(v) for k, v in PROPS.items()},
           "inside_tile": round(float(inside.mean()), 3), "ls_clear": round(float(np.nanmean(L[7] >= 0.8)), 3),
           "defective_share": round(float((D == 1).mean()), 4), "defective_is_nodata": None if np.isnan(def_nodata) else round(def_nodata, 3),
           "rel_diff_defective": None if np.isnan(rel_def) else round(rel_def, 3), "rel_diff_normal": round(rel_good, 3),
           "sec": round(time.time() - t0)}
    state = pd.concat([state[state["scene_id"] != sid], pd.DataFrame([row])], ignore_index=True)
    state.to_csv(sp, index=False); n_new += 1
    print(f"  {n + 1}/{len(en)} {sid[33:48]} {row['sec']} s | defective {row['defective_share']:.4f}, nodata in source {row['defective_is_nodata']}, "
          f"diff to neighbours: defective {row['rel_diff_defective']} vs normal {row['rel_diff_normal']}")

# ---- summary ----
st = state[state["fetch_version"] == FETCH_VERSION].copy()
pd.set_option("display.width", 250)
print(f"\nfetched this run: {n_new} in {(time.time() - t_all) / 60:.0f} min | total in format v{FETCH_VERSION}: {len(st)} of {len(en)}")
cols = ["sun_elev", "across_offnadir", "along_offnadir", "aot", "cloud_cover", "haze_cover", "ls_clear", "ls_dt"]
print("\nscene facts (median / min / max):\n" + st[cols].apply(pd.to_numeric, errors="coerce").describe().loc[["50%", "min", "max"]].round(2).to_string())
print(f"\nDEFECTIVE PIXELS: share of band-pixels {st['defective_share'].median():.4f} (median) | stored as nodata by DLR: "
      f"{st['defective_is_nodata'].median()} | difference to neighbours: defective {st['rel_diff_defective'].median()} vs normal {st['rel_diff_normal'].median()}")
print("   -> if 'nodata by DLR' ~ 0 and the defective difference is close to the normal one, DLR has REPAIRED them (values usable).")

## A5 Band check (evidence)
Mean spectra, pixel-level spectral roughness, spikes, low signal and defective share per band. It was run on the 5 pilot cells
(format v1, defective pixels as nodata); on the 40 cells in format v2 the counts differ slightly. Its result is fixed in A6.

In [ ]:
# A5 (= EN-05 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-05: EnMAP band check on the pilot cells: mean spectra, nodata / defective share per band, spikes, pixel-level noise,
#        low signal near the water-vapour gaps -> PROPOSED drop list (printed only; nothing is changed).
import os, glob, json, warnings
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"; PROC = f"{ROOT}/processed"
WL = np.array(json.load(open(f"{ROOT}/config/enmap_bands.json"))["wavelength_nm"]); NB = len(WL)
# ---- candidate rules (design choices, as for EMIT E-03 / Hyperion) ----
LOW, SPIKE, ROUGH, NODATA = 0.01, 0.30, 0.05, 0.05     # mean refl <= 0.01 | >30% off neighbours | pixel roughness > 5% | > 5% nodata

files = sorted(glob.glob(f"{PROC}/*_enmap.tif"))
S, ND, RG, ANYDEF = [], [], [], []
for f in files:
    with rasterio.open(f) as s: A = s.read()
    E, M = A[:NB].astype("float32"), A[NB:]
    inside = M[0] != 255
    clear = inside & (M[0] == 0) & (M[1] == 0) & (M[2] == 0) & (M[3] == 0) & (M[4] == 0) & (M[5] == 1)   # land, no cloud/shadow/haze/cirrus/snow
    nd = E == -32768
    ND.append(nd[:, inside].mean(axis=1)); ANYDEF.append(float(nd[:, inside].any(axis=0).mean()))
    R = np.where(nd, np.nan, E / 1e4)[:, clear]                                    # bands x clear pixels
    S.append(np.nanmean(R, axis=1))
    nbm = (R[:-2] + R[2:]) / 2                                                    # pixel-level spectral roughness
    rg = np.full(NB, np.nan); rg[1:-1] = np.nanmedian(np.abs(R[1:-1] - nbm) / np.maximum(np.abs(nbm), 1e-3), axis=1); RG.append(rg)
S, ND, RG = np.array(S), np.array(ND), np.array(RG)
print(f"pilot cells: {len(files)} | clear land pixels used per cell: see spectra | pixels with >= 1 defective band: "
      f"{[round(x, 3) for x in ANYDEF]}")

# ---- criteria per band ----
low = np.nanmedian(S, axis=0) <= LOW
spk = np.zeros(S.shape, bool)
for n in range(len(S)):
    s = S[n]; nb_ = (s[:-2] + s[2:]) / 2
    spk[n, 1:-1] = (nb_ > 0.02) & (np.abs(s[1:-1] - nb_) / nb_ > SPIKE)
spike = spk.sum(axis=0) >= 3
rough = np.nanmedian(RG, axis=0) > ROUGH
nod = ND.max(axis=0) > NODATA
cand = low | spike | rough | nod
def ranges(ix):
    o, s0, p = [], None, None
    for x in ix:
        if s0 is None: s0 = p = x
        elif x == p + 1: p = x
        else: o.append((s0, p)); s0 = p = x
    if s0 is not None: o.append((s0, p))
    return o
show = lambda m: ", ".join(f"b{a + 1}-{b + 1} ({WL[a]:.0f}-{WL[b]:.0f} nm)" for a, b in ranges(np.where(m)[0])) or "none"
print("\nPROPOSED DROP CANDIDATES (band numbers 1-224):")
print("  low signal (median mean refl <= 0.01):", show(low))
print("  spikes (in >= 3 of 5 cells):          ", show(spike))
print("  noisy (pixel roughness > 5%):          ", show(rough))
print("  nodata / defective > 5% (any cell):    ", show(nod))
print(f"  ALL candidates: {int(cand.sum())} bands -> keep {NB - int(cand.sum())} | {show(cand)}")

# ---- bands next to the two water-vapour gaps + the long-wave end (look for edge effects) ----
gaps = [i for i in range(NB - 1) if WL[i + 1] - WL[i] > 15]
print("\nbands next to the gaps and at the ends (median mean refl | median roughness):")
for i in gaps:
    for k in list(range(max(i - 3, 0), i + 1)) + list(range(i + 1, min(i + 5, NB))):
        print(f"   b{k + 1:3d} {WL[k]:7.1f} nm  refl {np.nanmedian(S[:, k]):.3f}  rough {np.nanmedian(RG[:, k]):.3f}")
    print("   ---")
for k in list(range(0, 3)) + list(range(NB - 5, NB)):
    print(f"   b{k + 1:3d} {WL[k]:7.1f} nm  refl {np.nanmedian(S[:, k]):.3f}  rough {np.nanmedian(RG[:, k]):.3f}")
pd.DataFrame({"band": np.arange(1, NB + 1), "wl_nm": WL.round(1), "median_refl": np.nanmedian(S, 0).round(4),
              "roughness": np.nanmedian(RG, 0).round(4), "max_nodata": ND.max(0).round(4), "spike_cells": spk.sum(0),
              "candidate": cand}).to_csv(f"{ROOT}/batch/enmap_band_check.csv", index=False)

# ---- figure: mean spectra + roughness, candidates shaded ----
fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
for n, f in enumerate(files): ax[0].plot(WL, S[n], lw=1, label=os.path.basename(f)[33:48])
ax[1].plot(WL, np.nanmedian(RG, 0), "k", lw=1); ax[1].axhline(ROUGH, color="r", ls="--", lw=.8)
for a, b in ranges(np.where(cand)[0]):
    for x in ax: x.axvspan(WL[a] - 3, WL[b] + 3, color="red", alpha=.15)
ax[0].set_ylabel("mean reflectance (clear land)"); ax[0].legend(fontsize=7); ax[1].set_ylabel("pixel roughness"); ax[1].set_xlabel("wavelength (nm)")
ax[1].set_ylim(0, 0.2); ax[0].set_title("EnMAP pilot cells: mean spectra; red = drop candidates"); plt.tight_layout(); plt.show()
print(f"saved: {ROOT}/batch/enmap_band_check.csv")

## A6 Final band list + defective-pixel fill test
Drops 47 bands for physical reasons (oxygen 764 nm, water vapour 920-1004 and 1116-1152 nm, empty 1343-1390 nm, the edges of the
1400 / 1900 nm gaps, 418-429 nm and 2415-2445 nm) -> **177 bands kept**. No further band has a median defective share above 5 %.
Defective pixels are filled from good 3x3 neighbours: tested on normal pixels the error is 5.2 %, the same as the normal difference
between neighbours; without filling 83 % of pixels would miss at least one band.

In [ ]:
# A6 (= EN-07 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-07: final EnMAP band list (fixed physical drops + bands with many defective pixels) and a TEST of filling the
#        remaining defective pixels from their 3x3 neighbours (accuracy measured on normal pixels). Writes config only.
import os, glob, json, warnings
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"; PROC = f"{ROOT}/processed"
cfgp = f"{ROOT}/config/enmap_bands.json"; cfg = json.load(open(cfgp)); WL = np.array(cfg["wavelength_nm"]); NB = len(WL)

# ---- design choices ----
FIXED_DROP = (list(range(1, 4)) + [62] + list(range(84, 103)) + list(range(112, 116)) + list(range(131, 140))
              + list(range(166, 172)) + list(range(220, 225)))                    # band numbers 1-224, from EN-05
MAX_DEF_SHARE = 0.05                                                              # drop a band if median defective share > 5 %
fixed = np.zeros(NB, bool); fixed[np.array(FIXED_DROP) - 1] = True

# ---- defective share per band and cell ----
fd = sorted(glob.glob(f"{PROC}/*_defective.tif")); DS = []
for f in fd:
    with rasterio.open(f.replace("_defective.tif", "_enmap.tif")) as s: inside = s.read(NB + 1) != 255
    with rasterio.open(f) as s: D = s.read() == 1
    DS.append(D[:, inside].mean(axis=1))
DS = np.array(DS); med, mx = np.median(DS, axis=0), DS.max(axis=0)
free = ~fixed
print(f"cells: {len(fd)} | fixed physical drops: {int(fixed.sum())} bands")
print("median defective share among the other bands, quantiles 50/75/90/95/99 %:", np.quantile(med[free], [.5, .75, .9, .95, .99]).round(4).tolist())
print("bands (outside the fixed drops) with median defective share above:", {t: int((med[free] > t).sum()) for t in (0.005, 0.01, 0.02, 0.05, 0.1)})
top = np.argsort(-np.where(free, med, -1))[:15]
print("\nworst 15 bands outside the fixed drops (band, nm, median share, max share):")
for k in top: print(f"   b{k + 1:3d} {WL[k]:7.1f} nm  median {med[k]:.4f}  max {mx[k]:.4f}")
defdrop = free & (med > MAX_DEF_SHARE)
keep = ~(fixed | defdrop)
print(f"\nextra drops (median defective share > {MAX_DEF_SHARE}): {[int(k) + 1 for k in np.where(defdrop)[0]] or 'none'}")
print(f"KEEP: {int(keep.sum())} of {NB} bands")

# ---- after the drops: how many pixels still have a defective band? + fill test ----
def fill(E, D):                                    # mean of good 3x3 neighbours in the same band
    good = ~D & np.isfinite(E)
    num = uniform_filter(np.where(good, E, 0).astype("float32"), size=(1, 3, 3))
    den = uniform_filter(good.astype("float32"), size=(1, 3, 3))
    est = np.where(den > 0, num / np.maximum(den, 1e-6), np.nan)
    return est, den
pix_any, errs, unfill, demo = [], [], [], None
for f in fd:
    with rasterio.open(f.replace("_defective.tif", "_enmap.tif")) as s:
        E = s.read(list(np.where(keep)[0] + 1)).astype("float32"); inside = s.read(NB + 1) != 255
    with rasterio.open(f) as s: D = s.read(list(np.where(keep)[0] + 1)) == 1
    E[E == -32768] = np.nan; E /= 1e4
    pix_any.append(float(D[:, inside].any(axis=0).mean()))
    est, den = fill(E, D)
    unfill.append(float((D & ~(den > 0))[:, inside].sum() / max(1, D[:, inside].sum())))
    g = ~D & np.isfinite(E) & inside[None]                      # test the fill on normal pixels: predict them from their neighbours
    good = g.copy(); num = uniform_filter(np.where(good, E, 0), size=(1, 3, 3)) * 9 - np.where(good, E, 0)
    cnt = uniform_filter(good.astype("float32"), size=(1, 3, 3)) * 9 - good
    pred = np.where(cnt > 0.5, num / np.maximum(cnt, 1e-6), np.nan)
    rel = np.abs(pred - E) / np.maximum(np.abs(E), 0.01)
    errs.append(float(np.nanmedian(rel[g][::20])))
    if demo is None and D.any():
        k = int(np.argmax(D.sum(axis=(1, 2)))); demo = (os.path.basename(f)[33:48], WL[np.where(keep)[0][k]], E[k], D[k], np.where(D[k], est[k], E[k]))
print(f"\nwith the kept bands: pixels with >= 1 defective band: median {np.median(pix_any):.3f} (max {np.max(pix_any):.3f})")
print(f"FILL TEST on normal pixels (3x3 neighbour mean): median relative error {np.median(errs):.3f} | defective pixels that cannot be filled "
      f"(no good neighbour): {np.median(unfill):.4f}")

cfg.update({"fixed_drop_band_numbers": FIXED_DROP, "defective_drop_band_numbers": [int(k) + 1 for k in np.where(defdrop)[0]],
            "keep_band_numbers": [int(k) + 1 for k in np.where(keep)[0]], "keep_band_indices": [int(k) for k in np.where(keep)[0]],
            "max_defective_share": MAX_DEF_SHARE, "defective_handling": "filled with the mean of good 3x3 neighbours in the same band",
            "note": "fixed drops from EN-05 (O2-A 764 nm, H2O 920-1004 / 1116-1152 nm, empty 1343-1390 nm, gap edges, ends); band number = EnMAP band 1-224"})
json.dump(cfg, open(cfgp, "w"))
print(f"saved: {cfgp} (keep {int(keep.sum())} bands)")

# ---- figure: defective share per band + one example of the fill ----
fig = plt.figure(figsize=(13, 7)); ax = fig.add_subplot(2, 1, 1)
ax.bar(WL, med, width=6, color=np.where(fixed, "lightgrey", np.where(defdrop, "red", "tab:blue")))
ax.axhline(MAX_DEF_SHARE, color="r", ls="--", lw=.8); ax.set_ylabel("median defective share"); ax.set_xlabel("wavelength (nm)")
ax.set_title("grey = fixed physical drops, red = dropped for defective pixels, blue = kept"); ax.set_ylim(0, max(0.12, float(np.nanmax(med[free])) * 1.1))
if demo:
    nm, w, e, d, filled = demo; lo, hi = np.nanpercentile(e, [2, 98])
    for j, (img, t) in enumerate([(e, f"{nm} band {w:.0f} nm (original)"), (d, "defective pixels"), (filled, "after 3x3 fill")]):
        a = fig.add_subplot(2, 3, 4 + j); a.imshow(img, vmin=(0 if j == 1 else lo), vmax=(1 if j == 1 else hi), cmap="gray"); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

# PART B: QC and handover
## B1 QC v2.3 -> pair folders
The same QC v2.3 as Hyperion and EMIT, plus the EnMAP-specific steps: defective pixels filled (3x3, then 5x5); EnMAP cloud, cloud shadow,
cirrus and snow masks applied (the EnMAP haze flag is recorded but not applied: it marked clear areas); then valid pixel (>= 80 % of
bands > 0), shift search +-8 px, residual bright cloud / shadow vs Landsat (2 px buffer), and the checks: joint coverage >= 0.5,
NIR ratio 0.85-1.15, NDVI r >= 0.7, AOD <= 0.8 (EnMAP sceneAOT / 1000), sun >= 25 deg, Landsat clear >= 0.85, no unresolved shift,
large shifts confirmed, NDVI gain >= 0.80. It also prints the NIR ratio against the viewing angle.

In [ ]:
# B1 (= EN-08 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-08: EnMAP QC v2.3 (same rules as Hyperion / EMIT v2.3) + EnMAP masks + defective-pixel fill
#        -> pairs/<N>/ (enmap_reflectance.tif, landsat_<id>.tif, pair_info.json) + pairs/enmap_final_qc_v2.3.csv. Safe to re-run.
import os, re, glob, json, shutil, warnings
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, binary_dilation, uniform_filter
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
PROC, PAIRS, BATCH = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch"
os.makedirs(PAIRS, exist_ok=True)

# ---- QC v2.3 rules (same as EMIT / Hyperion v2.3; design choices) ----
QC_VERSION = "v2.3"
MIN_COV, RLO, RHI = 0.5, 0.85, 1.15
MIN_NDVI_R, MAX_AOD, MIN_SUN, MIN_LS_CLEAR = 0.7, 0.8, 25.0, 0.85
MIN_POS_BANDS = 0.8
SHIFT_MAX, MIN_R_SHIFT, MIN_GAIN, BIG_SHIFT, CONFIRM_MIN = 8, 0.6, 0.03, 3, 2
K_RATIO, D_REF, MIN_LS_NIR, ABS_BLUE, DILATE_PX = 1.5, 0.06, 0.10, 0.35, 2
VEG_NDVI, MIN_NDVI_GAIN, MILD_NDVI_GAIN, MIN_VEG_PX = 0.3, 0.80, 0.85, 500
FLAG_BANDS = ("blue", "red", "swir1")
# ---- EnMAP specific (design choices) ----
MASKS = ["quality_cloud", "quality_cloud_shadow", "quality_haze", "quality_cirrus", "quality_snow", "quality_classes"]   # order in the file
APPLY = ["quality_cloud", "quality_cloud_shadow", "quality_cirrus", "quality_snow"]   # haze flag recorded, NOT applied
WIN = {"blue": (452, 512), "green": (533, 590), "red": (636, 673), "nir": (851, 879), "swir1": (1566, 1651), "swir2": (2107, 2294)}
NAMES = list(WIN)
fx = lambda x, d=3: None if x is None or not np.isfinite(x) else round(float(x), d)
pd.set_option("display.width", 260)

cfg = json.load(open(f"{ROOT}/config/enmap_bands.json"))
keep = cfg["keep_band_indices"]; wl_all = np.array(cfg["wavelength_nm"]); wlk = wl_all[keep]; NB = len(wl_all); nk = len(keep)
st = pd.read_csv(f"{BATCH}/enmap_fetch_state.csv"); st = st[st["fetch_version"] == 2].drop_duplicates("scene_id").reset_index(drop=True)
ipath = f"{PAIRS}/pair_index.csv"
idx = pd.read_csv(ipath) if os.path.exists(ipath) else pd.DataFrame(columns=["pair_no", "scene_id", "cell_id", "profile", "landsat_id"])
def cgroup(k):
    if k is None or pd.isna(k): return None
    k = int(k); return "tropical" if k <= 3 else "arid" if k <= 7 else "temperate" if k <= 16 else "cold" if k <= 28 else "polar"

# ---- helpers (as EMIT B1) ----
def sh(A, dy, dx): return A[max(dy, 0):A.shape[0] + min(dy, 0), max(dx, 0):A.shape[1] + min(dx, 0)]
def pearson(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() >= 200 else np.nan
def shift_cube(A, dy, dx):
    out = np.full_like(A, np.nan); hh, ww = A.shape[-2:]
    ys, ye, xs, xe = max(0, -dy), min(hh, hh - dy), max(0, -dx), min(ww, ww - dx)
    out[..., ys:ye, xs:xe] = A[..., ys + dy:ye + dy, xs + dx:xe + dx]; return out
def win_mean(H, k):
    sel = (wlk >= WIN[k][0]) & (wlk <= WIN[k][1])
    return np.nanmean(H[sel], axis=0) if sel.any() else np.full(H.shape[1:], np.nan)
def ndvi_pair(H, SR, clear):
    rd, ni = win_mean(H, "red"), win_mean(H, "nir")
    nh = (ni - rd) / (ni + rd); nl = (SR[3] - SR[2]) / (SR[3] + SR[2])
    u = (np.isfinite(nh) & np.isfinite(nl) & clear & (rd > 0) & (ni > 0) & (SR[2] > 0) & (SR[3] > 0) & (np.abs(nl) <= 1))
    return np.where(u, nh, np.nan), np.where(u, nl, np.nan)
def best_shift(nh, nl):
    r0 = pearson(nh, nl); best = (r0, (0, 0))
    for dy in range(-SHIFT_MAX, SHIFT_MAX + 1):
        for dx in range(-SHIFT_MAX, SHIFT_MAX + 1):
            r = pearson(sh(nh, dy, dx), sh(nl, -dy, -dx))
            if np.isfinite(r) and (not np.isfinite(best[0]) or r > best[0]): best = (r, (dy, dx))
    return r0, best
def confirm(H, SR, clear):
    sig = {"red": (win_mean(H, "red"), SR[2]), "swir1": (win_mean(H, "swir1"), SR[4]),
           "bright": (np.nanmean([win_mean(H, k) for k in NAMES], axis=0), np.nanmean(SR, axis=0))}
    det, n_ok = {}, 0
    for k, (a, b) in sig.items():
        a = np.where(clear & np.isfinite(a), a, np.nan); b = np.where(clear & np.isfinite(b) & (b > 0), b, np.nan)
        _, (rb, s) = best_shift(a, b); ok = max(abs(s[0]), abs(s[1])) <= 1
        det[k] = [int(s[0]), int(s[1])]; n_ok += ok
    return n_ok, det
def med5(a): return median_filter(np.where(np.isfinite(a), a, 0).astype("float32"), size=5)
def mratio(a, b, m):
    m = m & np.isfinite(a) & np.isfinite(b) & (b > 0)
    return float(np.median(a[m]) / np.median(b[m])) if m.sum() >= 1000 else np.nan
def cloud_mask(H, SR, clear):
    fin = np.isfinite(H).any(axis=0); joint = fin & clear
    Eb, En = win_mean(H, "blue"), win_mean(H, "nir")
    kb, kn = mratio(Eb, SR[0], joint), mratio(En, SR[3], joint)
    absc = fin & ~clear & (Eb > ABS_BLUE)
    if not (np.isfinite(kb) and np.isfinite(kn)): z = np.zeros_like(fin); return z, z, absc
    bright = joint & (med5(Eb - kb * SR[0]) > D_REF) & (Eb > K_RATIO * kb * SR[0])
    shadow = joint & (SR[3] > MIN_LS_NIR) & (med5(En - kn * SR[3]) < -D_REF) & (En < kn * SR[3] / K_RATIO)
    return (binary_dilation(bright, iterations=DILATE_PX) & fin, binary_dilation(shadow, iterations=DILATE_PX) & fin,
            binary_dilation(absc, iterations=DILATE_PX) & fin)
def ndvi_gain(nh, nl):
    v = np.isfinite(nh) & np.isfinite(nl) & (nl > VEG_NDVI)
    return float(np.mean(nh[v]) / np.mean(nl[v])) if v.sum() >= MIN_VEG_PX else np.nan
def fill_defective(E, D):                   # NEW (EnMAP): mean of good neighbours, 3x3 then 5x5; rest -> NaN
    out, rem = E.copy(), D.copy(); good = ~D & np.isfinite(E)
    for size in (3, 5):
        num = uniform_filter(np.where(good, E, 0).astype("float32"), size=(1, size, size))
        den = uniform_filter(good.astype("float32"), size=(1, size, size))
        m = rem & (den > 1e-6); out[m] = (num / np.maximum(den, 1e-6))[m]; rem &= ~m
    out[rem] = np.nan
    return out, rem

rows, viz = [], {}
for n, m in st.iterrows():
    sid = m["scene_id"]; fe, fd = f"{PROC}/{sid}_enmap.tif", f"{PROC}/{sid}_defective.tif"; lid = m["ls_id"]
    fl = f"{PROC}/{sid}_landsat.tif"
    if not all(os.path.exists(x) for x in (fe, fd, fl)): rows.append({"pair": None, "scene": sid[33:48], "status": "FAIL", "reasons": "files missing"}); continue
    with rasterio.open(fe) as s:
        R = s.read([k + 1 for k in keep]).astype("float32"); MK = s.read(list(range(NB + 1, NB + len(MASKS) + 1))); tf, crs = s.transform, s.crs
    with rasterio.open(fd) as s: D = s.read([k + 1 for k in keep]) == 1
    with rasterio.open(fl) as s: L = s.read().astype("float32")
    R = np.where(R == -32768, np.nan, R / 1e4)
    inside = MK[0] != 255
    H, unfilled = fill_defective(R, D)
    filled_frac = float(D[:, inside].mean()); unfilled_frac = float(unfilled[:, inside].mean())
    emask = np.zeros(inside.shape, bool)
    for k in APPLY: emask |= MK[MASKS.index(k)] == 1
    valid = inside & ~emask & ((H > 0).mean(axis=0) >= MIN_POS_BANDS)
    H[:, ~valid] = np.nan
    SR, cf = L[:6], L[7]; ls_has = np.isfinite(cf); clear = ls_has & np.isfinite(SR[3]) & (cf >= 0.8)
    mfr = {k.replace("quality_", ""): fx(float((MK[MASKS.index(k)] == 1)[inside].mean())) for k in MASKS[:5]}
    mfr["water"] = fx(float((MK[5] == 2)[inside].mean()))

    # registration (+-8 px)
    nh, nl = ndvi_pair(H, SR, clear); r0, (rb, (dy, dx)) = best_shift(nh, nl)
    if (dy, dx) != (0, 0) and np.isfinite(rb) and rb > MIN_R_SHIFT and (rb - r0) > MIN_GAIN: H = shift_cube(H, dy, dx)
    else: dy, dx = 0, 0
    # our residual cloud / shadow mask (relative to Landsat), then everything after masking
    fin0 = np.isfinite(H).any(axis=0); nj = max(1, int((fin0 & clear).sum()))
    b_m, s_m, a_m = cloud_mask(H, SR, clear); H[:, b_m | s_m | a_m] = np.nan
    cfr = {"bright_cloud": fx((b_m & clear).sum() / nj), "cloud_shadow": fx((s_m & clear).sum() / nj),
           "bright_where_landsat_not_clear": fx(a_m.sum() / max(1, int(fin0.sum())))}
    nh2, nl2 = ndvi_pair(H, SR, clear); r_after, (r_res, sh_res) = best_shift(nh2, nl2)
    at_edge = SHIFT_MAX in (abs(dy), abs(dx))
    unresolved = at_edge or (sh_res != (0, 0) and np.isfinite(r_res) and (r_res - r_after) > MIN_GAIN)
    big = max(abs(dy), abs(dx)) > BIG_SHIFT
    n_conf, conf_det = confirm(H, SR, clear) if big else (None, None)
    big_ok = (not big) or (n_conf >= CONFIRM_MIN)
    gain = ndvi_gain(nh2, nl2); mild = bool(np.isfinite(gain) and MIN_NDVI_GAIN <= gain < MILD_NDVI_GAIN)

    fin = np.isfinite(H).any(axis=0)
    cov_enmap = float((fin & inside).sum() / max(1, inside.sum()))
    ls_clear = float((cf[fin & ls_has] >= 0.8).mean()) if (fin & ls_has).any() else np.nan
    cov = float((fin & clear).mean())
    ratio = {}
    for i, k in enumerate(NAMES):
        a_ = win_mean(H, k); b_ = SR[i]; mm = np.isfinite(a_) & clear & (b_ > 0)
        ratio[k] = float(np.median(a_[mm]) / np.median(b_[mm])) if mm.sum() >= 1000 else np.nan
    flag_ok = {k: bool(RLO <= ratio[k] <= RHI) for k in FLAG_BANDS}
    aot = float(m["aot"]) if pd.notna(m["aot"]) else np.nan; aod = aot / 1000 if aot > 5 else aot    # sceneAOT stored x1000 (my reading)
    sun = float(m["sun_elev"])
    checks = {"coverage": cov >= MIN_COV, "nir_ratio": bool(RLO <= ratio["nir"] <= RHI), "no_unresolved_shift": not unresolved,
              "ndvi_r": bool(np.isfinite(r_after) and r_after >= MIN_NDVI_R), "aod": bool(not np.isfinite(aod) or aod <= MAX_AOD),
              "sun_elev": sun >= MIN_SUN, "landsat_clear": bool(np.isfinite(ls_clear) and ls_clear >= MIN_LS_CLEAR),
              "large_shift_confirmed": bool(big_ok), "haze_ndvi_gain": bool(not np.isfinite(gain) or gain >= MIN_NDVI_GAIN)}
    status = "PASS" if all(checks.values()) else "FAIL"; reasons = ",".join(k for k, v in checks.items() if not v)
    flags_out = ",".join([k for k, v in flag_ok.items() if not v] + (["haze_mild"] if mild else [])
                         + (["haze_unchecked"] if not np.isfinite(gain) else []))

    # pair number (append-only) + folder
    hit = idx[idx["scene_id"] == sid]
    if len(hit): no = int(hit.iloc[0]["pair_no"])
    else:
        no = int(idx["pair_no"].max()) + 1 if len(idx) else 1
        idx.loc[len(idx)] = [no, sid, m["cell_id"], m["profile"], lid]; idx.to_csv(ipath, index=False)
    folder = f"{PAIRS}/{no}"; os.makedirs(folder, exist_ok=True); jp = f"{folder}/pair_info.json"
    prev_status = json.load(open(jp))["final_qc"]["status"] if os.path.exists(jp) else None
    out = np.full(H.shape, -32768, "int16"); ok = np.isfinite(H); out[ok] = np.clip(np.round(H[ok] * 10000), -32000, 32000).astype("int16")
    with rasterio.open(f"{folder}/enmap_reflectance.tif", "w", driver="GTiff", height=out.shape[1], width=out.shape[2], count=nk,
                       dtype="int16", crs=crs, transform=tf, nodata=-32768, compress="deflate", tiled=True) as dst:
        dst.write(out)
        for k, (b, w) in enumerate(zip(keep, wlk), start=1): dst.set_band_description(k, f"{w:.1f} nm (EnMAP b{int(b) + 1:03d})")
        dst.update_tags(scale_factor_inverse=10000, units="surface reflectance x 10000", source=sid, product="EnMAP L2A (DLR)",
                        proc_version=str(m["version"]), defective_pixels="filled with the mean of good 3x3 (then 5x5) neighbours",
                        valid_pixel="EnMAP cloud/shadow/cirrus/snow = 0, >= 80% bands > 0, minus residual bright cloud / shadow vs Landsat (2 px buffer)",
                        shift_applied_yx=str((dy, dx)), qc_version=QC_VERSION)
    with rasterio.open(f"{folder}/landsat_{lid}.tif", "w", driver="GTiff", height=L.shape[1], width=L.shape[2], count=L.shape[0],
                       dtype="float32", crs=crs, transform=tf, nodata=np.nan, compress="deflate") as dst:
        dst.write(L)
        for k, nme in enumerate(["blue", "green", "red", "nir", "swir1", "swir2", "LST_K", "clear_frac"], start=1): dst.set_band_description(k, nme)
        dst.update_tags(note="Landsat bilinear-resampled onto the EnMAP 30 m grid; clear_frac = clear share; NaN = no data")
    kg = m["kg"]
    info = {"pair_no": no, "enmap_scene": sid, "enmap_date": str(m["date"]), "landsat_scene": lid, "landsat_minus_enmap_days": fx(float(m["ls_dt"]), 4),
            "cell_id": m["cell_id"], "cell_lon": float(m["lon"]), "cell_lat": float(m["lat"]), "profile": m["profile"],
            "koppen_code": int(kg) if pd.notna(kg) else None, "climate_group": cgroup(kg), "status": status, "qc_version": QC_VERSION,
            "enmap_proc_version": str(m["version"]),
            "geometry": {"sun_elev": fx(sun, 2), "sun_az": fx(float(m["sun_az"]), 2), "across_offnadir": fx(float(m["across_offnadir"]), 2),
                         "along_offnadir": fx(float(m["along_offnadir"]), 2)},
            "atmosphere": {"aod550_scene": fx(aod, 3), "scene_wv_raw": m["wv"]},
            "final_qc": {"status": status, "failed_checks": reasons, "flags_outside_range": flags_out, "swir1_flag_ok": flag_ok["swir1"],
                         "coverage_joint": fx(cov), "coverage_enmap": fx(cov_enmap), "landsat_clear_frac": fx(ls_clear), "sun_elev": fx(sun, 1),
                         "aod550": fx(aod, 2), "ndvi_r_before": fx(r0), "ndvi_r_after": fx(r_after), "ndvi_gain": fx(gain), "haze_mild": mild,
                         "enmap_mask_frac": mfr, "enmap_masks_applied": APPLY, "residual_cloud_masked_frac": cfr,
                         "defective_filled_frac": fx(filled_frac, 4), "defective_unfilled_frac": fx(unfilled_frac, 5),
                         "applied_shift_yx": [dy, dx], "large_shift": big, "large_shift_confirming_signals": n_conf, "large_shift_signal_peaks": conf_det,
                         "shift_at_search_edge": at_edge, "ratio_enmap_over_landsat": {k: fx(v, 2) for k, v in ratio.items()},
                         "n_bands": nk, "keep_band_numbers": cfg["keep_band_numbers"],
                         "thresholds": {"min_cov_joint": MIN_COV, "ratio": [RLO, RHI], "max_aod": MAX_AOD, "min_sun": MIN_SUN,
                                        "min_ndvi_r": MIN_NDVI_R, "min_landsat_clear": MIN_LS_CLEAR, "shift_search_px": SHIFT_MAX,
                                        "shift_min_r": MIN_R_SHIFT, "shift_min_gain": MIN_GAIN, "large_shift_px": BIG_SHIFT,
                                        "large_shift_min_confirming": CONFIRM_MIN,
                                        "cloud_mask": {"ratio_factor": K_RATIO, "min_diff": D_REF, "shadow_min_landsat_nir": MIN_LS_NIR,
                                                       "abs_blue_where_landsat_not_clear": ABS_BLUE, "median_window": 5, "dilation_px": DILATE_PX},
                                        "haze": {"veg_landsat_ndvi": VEG_NDVI, "min_ndvi_gain": MIN_NDVI_GAIN, "mild_below": MILD_NDVI_GAIN,
                                                 "min_veg_px": MIN_VEG_PX}, "flag_bands": list(FLAG_BANDS)}}}
    json.dump(info, open(jp, "w"), indent=1, default=str)
    rows.append({"pair": no, "date": m["date"], "cell": m["cell_id"], "climate": cgroup(kg), "sun": fx(sun, 1), "offnadir": fx(float(m["across_offnadir"]), 1),
                 "aod": fx(aod, 2), "cov": fx(cov, 2), "ls_clear": fx(ls_clear, 2),
                 "enmap_cloud%": fx(100 * (mfr["cloud"] or 0), 1), "haze_flag%": fx(100 * (mfr["haze"] or 0), 1),
                 "our_cloud%": fx(100 * ((cfr["bright_cloud"] or 0) + (cfr["cloud_shadow"] or 0)), 1), "filled%": fx(100 * filled_frac, 2),
                 "gain": fx(gain, 2), "shift": f"{dy},{dx}", "confirm": n_conf, "ndvi_r": f"{fx(r0, 2)}->{fx(r_after, 2)}",
                 **{k: fx(ratio[k], 2) for k in NAMES}, "prev": prev_status, "status": status, "reasons": reasons, "flags": flags_out,
                 "big": big, "mild": mild})
    ib = [int(np.argmin(np.abs(wlk - w))) for w in (650, 550, 470)]
    rgb = np.nan_to_num(np.clip(np.stack([H[i] for i in ib], -1) / 0.3, 0, 1)); ov = rgb.copy()
    ov[MK[2] == 1] = [1, 0, 1]; ov[(b_m | a_m)] = [1, 0, 0]; ov[s_m] = [0, 1, 1]
    viz[no] = (rgb, ov, np.nan_to_num(np.clip(np.stack([SR[2], SR[1], SR[0]], -1) / 0.3, 0, 1)), nh2, nl2, status, reasons, gain)
    print(f"  {n + 1}/{len(st)} pair {no}: {status} {reasons}")

# ---- summary ----
res = pd.DataFrame(rows).sort_values("pair").reset_index(drop=True)
res.to_csv(f"{PAIRS}/enmap_final_qc_{QC_VERSION}.csv", index=False)
ok_rows = res[res["pair"].notna()]
print(f"\nQC {QC_VERSION} | pairs: {len(ok_rows)} | PASS: {int((ok_rows['status'] == 'PASS').sum())} | FAIL: {int((ok_rows['status'] == 'FAIL').sum())}")
print("\n" + ok_rows.drop(columns=["big", "mild", "prev"]).to_string(index=False))
pp = ok_rows[ok_rows["status"] == "PASS"]
print(f"\nPASS cells: {pp['cell'].nunique()} | PASS by climate group: {pp['climate'].value_counts(dropna=False).to_dict()}")
fails = ok_rows[ok_rows["status"] == "FAIL"].assign(reason=lambda x: x["reasons"].str.split(",")).explode("reason")
print("failed checks (a pair can fail several):", fails["reason"].value_counts().to_dict())
print("flags among PASS pairs:", {k: int(pp["flags"].fillna("").str.contains(k).sum()) for k in FLAG_BANDS + ("haze_mild",)})
print("median ratios over PASS pairs:", pp[NAMES].median().round(2).to_dict())
print("large shifts:", ok_rows.loc[ok_rows["big"] == True, ["pair", "shift", "confirm", "status"]].values.tolist() or "none")
x = ok_rows[["nir", "offnadir"]].dropna()
print(f"\nNIR question: Spearman(NIR ratio, |off-nadir|) = {fx(x['nir'].corr(x['offnadir'].abs(), method='spearman'), 2)} | "
      f"with the signed angle = {fx(x['nir'].corr(x['offnadir'], method='spearman'), 2)} | n = {len(x)}")

# ---- figure: FAIL pairs first, then haze_mild / large shifts (max 8). red = our bright cloud, cyan = our shadow, magenta = EnMAP haze flag (not applied)
show = (ok_rows[ok_rows["status"] == "FAIL"]["pair"].tolist() + ok_rows[(ok_rows["status"] == "PASS") & (ok_rows["mild"] | ok_rows["big"])]["pair"].tolist())[:8]
if show:
    fig, axs = plt.subplots(len(show), 4, figsize=(15, 3.6 * len(show)), squeeze=False)
    for i, no in enumerate(show):
        rgb, ov, lrgb, a2, b2, stt, rsn, g = viz[int(no)]
        axs[i, 0].imshow(rgb); axs[i, 0].set_title(f"pair {int(no)} EnMAP {stt} {rsn[:40]}", fontsize=8)
        axs[i, 1].imshow(ov); axs[i, 1].set_title("red/cyan = our masks, magenta = EnMAP haze flag", fontsize=8)
        axs[i, 2].imshow(lrgb); axs[i, 2].set_title("Landsat RGB", fontsize=8)
        mk = np.isfinite(a2) & np.isfinite(b2); sel = np.flatnonzero(mk.ravel())[::max(1, mk.sum() // 3000)]
        axs[i, 3].scatter(b2.ravel()[sel], a2.ravel()[sel], s=2, alpha=.4); axs[i, 3].plot([0, 1], [0, 1], "k--", lw=.8)
        axs[i, 3].set_xlim(-0.2, 1); axs[i, 3].set_ylim(-0.2, 1); axs[i, 3].set_title(f"NDVI gain {fx(g, 2)}", fontsize=8)
        for a in axs[i, :3]: a.axis("off")
    plt.tight_layout(); plt.show()
print(f"\nsaved: {PAIRS}/enmap_final_qc_{QC_VERSION}.csv")

## B2 Landsat scene edge: use the neighbouring Landsat scene
For pairs with Landsat clear < 0.85: tells whether the cell lies outside the screening's Landsat scene or under cloud, and finds the best
Landsat 8/9 scene within +-15 days. On 24 Oct all 7 such pairs were **scene-edge** cases (straight edges, not clouds); 5 were fixed with
the neighbouring scene of the same overpass, pair 19 with +10 d and pair 21 with -14 d. Applied once (`APPLY = True`), then B1 was re-run.
`APPLY` is False here, so running it again only reports.

In [ ]:
# B2 (= EN-08b in the working notebook ENMAP_LANDSAT.ipynb)
# EN-08b: pairs with little clear Landsat: outside the Landsat scene or cloudy? + search a better Landsat 8/9 scene (+-15 days).
#         APPLY = False: report only. APPLY = True: replace the Landsat chip (old copy kept as _landsat_v1.tif), then re-run EN-08.
import os, re, glob, json, shutil
import numpy as np, pandas as pd, rasterio, ee
import matplotlib.pyplot as plt

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
PROC, PAIRS, BATCH = f"{ROOT}/processed", f"{ROOT}/pairs", f"{ROOT}/batch"
try: ee.Initialize(project="hyperspectral-thermal-511303")
except Exception: ee.Authenticate(); ee.Initialize(project="hyperspectral-thermal-511303")

APPLY = False                    # set True only after looking at the table
MAX_DT, MIN_CLEAR, HALF = 15, 0.85, 0.035
LSBANDS = ["blue", "green", "red", "nir", "swir1", "swir2", "LST_K", "clear_frac"]
NB = 224
qc = pd.read_csv(f"{PAIRS}/enmap_final_qc_v2.3.csv")
pidx = pd.read_csv(f"{PAIRS}/pair_index.csv").set_index("pair_no")
sp = f"{BATCH}/enmap_fetch_state.csv"; st = pd.read_csv(sp)
ho = pd.read_csv(f"{BASE}/handover_pairs.csv"); ho = ho[ho["sensor"].str.upper() == "ENMAP"].set_index("scene_id")

def landsat_img(lid):
    coll = "LANDSAT/LC09/C02/T1_L2" if lid.startswith("LC09") else "LANDSAT/LC08/C02/T1_L2"
    L = ee.Image(f"{coll}/{lid}"); qa = L.select("QA_PIXEL")
    clear = qa.bitwiseAnd(0b11110).eq(0).And(qa.neq(0))
    sr = L.select(["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]).multiply(0.0000275).add(-0.2).rename(LSBANDS[:6])
    lst = L.select("ST_B10").multiply(0.00341802).add(149.0).rename("LST_K")
    return (sr.addBands(lst).updateMask(clear).resample("bilinear")
              .addBands(clear.rename("clear_frac").toFloat().resample("bilinear")).toFloat().unmask(-9999))
def chip(img, bands, crs_epsg, tf, w, h):
    grid = {"dimensions": {"width": w, "height": h}, "crsCode": f"EPSG:{crs_epsg}",
            "affineTransform": {"scaleX": tf.a, "shearX": tf.b, "translateX": tf.c, "shearY": tf.d, "scaleY": tf.e, "translateY": tf.f}}
    arr = ee.data.computePixels({"expression": img.select(bands), "fileFormat": "NUMPY_NDARRAY", "grid": grid})
    A = np.stack([arr[b] for b in bands]).astype("float32"); A[A == -9999] = np.nan; return A

bad = qc[qc["pair"].notna() & (qc["ls_clear"].fillna(0) < MIN_CLEAR)].copy()
print(f"pairs with Landsat clear < {MIN_CLEAR}: {len(bad)} -> {bad['pair'].astype(int).tolist()}\n")
rows, figs = [], []
for _, q in bad.iterrows():
    no = int(q["pair"]); sid = pidx.loc[no, "scene_id"]; m = st[(st["scene_id"] == sid) & (st["fetch_version"] == 2)].iloc[0]
    with rasterio.open(f"{PROC}/{sid}_enmap.tif") as s:
        tf, epsg, w, h = s.transform, s.crs.to_epsg(), s.width, s.height
        inside = s.read(NB + 1) != 255; e = s.read(int(np.argmin(np.abs(np.array(json.load(open(f'{ROOT}/config/enmap_bands.json'))['wavelength_nm']) - 865))) + 1)
    ev = inside & (e != -32768)
    with rasterio.open(f"{PROC}/{sid}_landsat.tif") as s: cf = s.read(8)
    has = np.isfinite(cf)
    t = pd.Timestamp(m["date"]); cell = ee.Geometry.Rectangle([m["lon"] - HALF, m["lat"] - HALF, m["lon"] + HALF, m["lat"] + HALF])
    try: cc = ee.Image(f"LANDSAT/{m['ls_id'][:4]}/C02/T1_L2/{m['ls_id']}").get("CLOUD_COVER").getInfo()
    except Exception: cc = None
    # candidates: Landsat 8 + 9 scenes covering the cell within +-MAX_DT days
    cands = []
    for coll in ("LANDSAT/LC08/C02/T1_L2", "LANDSAT/LC09/C02/T1_L2"):
        c = ee.ImageCollection(coll).filterBounds(cell).filterDate((t - pd.Timedelta(days=MAX_DT)).strftime("%Y-%m-%d"),
                                                                  (t + pd.Timedelta(days=MAX_DT + 1)).strftime("%Y-%m-%d"))
        cands += c.aggregate_array("system:index").getInfo()
    best = None
    for lid in cands:
        try: cfc = chip(landsat_img(lid), ["clear_frac"], epsg, tf, w, h)[0]
        except Exception: continue
        clr = float(np.mean((cfc >= 0.8)[ev])) if ev.any() else 0.0
        dt = (pd.Timestamp(lid[-8:]) - t.normalize()).days
        cands_row = (lid, dt, round(clr, 3))
        if best is None or (clr, -abs(dt)) > (best[2], -abs(best[1])): best = cands_row
    rows.append({"pair": no, "date": m["date"], "cell": m["cell_id"], "screening_ls_clear": ho["ls_clear"].get(sid),
                 "landsat": m["ls_id"], "scene_cloud%": cc, "no_landsat_data": round(float((~has)[ev].mean()), 2),
                 "cloudy": round(float((has & (cf < 0.8))[ev].mean()), 2), "clear": round(float((cf >= 0.8)[ev].mean()), 2),
                 "n_candidates": len(cands), "best_alternative": best[0] if best else None, "alt_dt_days": best[1] if best else None,
                 "alt_clear": best[2] if best else None})
    figs.append((no, ev, cf))
    if APPLY and best and best[2] >= MIN_CLEAR and best[0] != m["ls_id"]:
        L = chip(landsat_img(best[0]), LSBANDS, epsg, tf, w, h); fl = f"{PROC}/{sid}_landsat.tif"
        if not os.path.exists(fl.replace("_landsat.tif", "_landsat_v1.tif")): shutil.copy(fl, fl.replace("_landsat.tif", "_landsat_v1.tif"))
        with rasterio.open(fl, "w", driver="GTiff", height=h, width=w, count=8, dtype="float32", crs=f"EPSG:{epsg}", transform=tf,
                           nodata=np.nan, compress="deflate") as dst:
            dst.write(L)
            for k, nme in enumerate(LSBANDS, start=1): dst.set_band_description(k, nme)
            dst.update_tags(landsat_scene=best[0], replaced=f"was {m['ls_id']} (EN-08b)", note="Landsat bilinear-resampled onto the EnMAP 30 m grid")
        old = f"{PAIRS}/{no}/landsat_{m['ls_id']}.tif"
        if os.path.exists(old): os.rename(old, f"{PAIRS}/{no}/replaced_landsat_{m['ls_id']}.tif.bak")   # not delivered (pattern landsat_*.tif)
        i = st.index[(st["scene_id"] == sid) & (st["fetch_version"] == 2)][0]
        st.loc[i, ["ls_id", "ls_dt", "ls_replaced_from"]] = [best[0], float(best[1]), m["ls_id"]]
        print(f"  pair {no}: Landsat replaced {m['ls_id']} -> {best[0]} (dt {best[1]} d, clear {best[2]})")
if APPLY: st.to_csv(sp, index=False)

res = pd.DataFrame(rows); pd.set_option("display.width", 250)
print(res.to_string(index=False))
ok = res[(res["alt_clear"].fillna(0) >= MIN_CLEAR) & (res["alt_dt_days"].abs() <= MAX_DT)]
print(f"\nrescuable with another Landsat scene (clear >= {MIN_CLEAR}, |dt| <= {MAX_DT} d): {len(ok)} {ok['pair'].tolist()}"
      + ("" if APPLY else "  -> set APPLY = True and re-run, then re-run EN-08"))
fig, axs = plt.subplots(1, len(figs), figsize=(3 * len(figs), 3.2), squeeze=False)
for a, (no, ev, cf) in zip(axs[0], figs):
    img = np.zeros(ev.shape + (3,)); img[ev] = [0.5, 0.5, 0.5]; img[np.isfinite(cf) & (cf >= 0.8)] += [0, 0.4, 0]; img[np.isfinite(cf) & (cf < 0.8)] = [0.8, 0, 0]
    a.imshow(np.clip(img, 0, 1)); a.set_title(f"pair {no}", fontsize=9); a.axis("off")
fig.suptitle("grey = EnMAP valid, green = Landsat clear, red = Landsat cloudy, black = no data", fontsize=9); plt.tight_layout(); plt.show()

## B3 Handover to PASSED-PAIRS
Copies the PASS pairs, moves stale copies to `_to_delete/`, writes `handover_pass.csv`, `enmap_bands.json` and `README.txt`.

In [ ]:
# B3 (= EN-09 in the working notebook ENMAP_LANDSAT.ipynb)
# EN-09: copy EnMAP PASS pairs (QC v2.3) into ENMAP-LANDSAT/PASSED-PAIRS (+ handover_pass.csv + README). Safe to re-run; nothing is deleted.
import os, glob, json, shutil, time
import pandas as pd

# ---- idempotent setup ----
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"; ROOT = f"{BASE}/ENMAP-LANDSAT"
PAIRS, OUT = f"{ROOT}/pairs", f"{ROOT}/PASSED-PAIRS"; os.makedirs(OUT, exist_ok=True)
QC_VERSION = "v2.3"
EXCLUDE = []                                                     # pair numbers to leave out after a visual check
DELIVER = ("enmap_reflectance.tif", "landsat_*.tif", "pair_info.json")

# ---- 1) status of every pair folder ----
pidx = pd.read_csv(f"{PAIRS}/pair_index.csv").set_index("pair_no")
folders = sorted([d for d in glob.glob(f"{PAIRS}/*") if os.path.isdir(d) and os.path.basename(d).isdigit()], key=lambda d: int(os.path.basename(d)))
rows, problems = [], []
for d in folders:
    no = int(os.path.basename(d)); jp = f"{d}/pair_info.json"
    if not os.path.exists(jp): problems.append(f"pair {no}: no pair_info.json"); continue
    info = json.load(open(jp)); q = info.get("final_qc") or {}; r = q.get("ratio_enmap_over_landsat") or {}
    if info.get("qc_version") != QC_VERSION: problems.append(f"pair {no}: qc_version {info.get('qc_version')} (run EN-08 first)")
    fl = q.get("flags_outside_range") or ""; g = info.get("geometry") or {}; cm = q.get("residual_cloud_masked_frac") or {}; em = q.get("enmap_mask_frac") or {}
    nls = len(glob.glob(f"{d}/landsat_*.tif"))
    if nls != 1: problems.append(f"pair {no}: {nls} landsat_*.tif files in the pair folder")
    rows.append({"pair": no, "status": info.get("status"), "qc_version": info.get("qc_version"), "failed_checks": q.get("failed_checks"),
                 "profile": info.get("profile"), "koppen_code": info.get("koppen_code"), "climate_group": info.get("climate_group"),
                 "cell_id": info.get("cell_id"), "cell_lon": info.get("cell_lon"), "cell_lat": info.get("cell_lat"),
                 "enmap_scene": info.get("enmap_scene"), "enmap_date": info.get("enmap_date"), "enmap_proc_version": info.get("enmap_proc_version"),
                 "landsat_scene": info.get("landsat_scene"), "landsat_minus_enmap_days": info.get("landsat_minus_enmap_days"),
                 "sun_elev": g.get("sun_elev"), "across_offnadir": g.get("across_offnadir"), "aod550": q.get("aod550"),
                 "coverage_joint": q.get("coverage_joint"), "landsat_clear_frac": q.get("landsat_clear_frac"),
                 "ndvi_r_after": q.get("ndvi_r_after"), "ndvi_gain": q.get("ndvi_gain"), "haze_mild": bool(q.get("haze_mild") or False),
                 "enmap_cloud_frac": em.get("cloud"), "enmap_haze_flag_frac": em.get("haze"),
                 "our_cloud_masked_frac": (cm.get("bright_cloud") or 0) + (cm.get("cloud_shadow") or 0),
                 "defective_filled_frac": q.get("defective_filled_frac"), "applied_shift_yx": q.get("applied_shift_yx"), "large_shift": q.get("large_shift"),
                 **{f"ratio_{k}": r.get(k) for k in ("blue", "green", "red", "nir", "swir1", "swir2")},
                 "blue_ok": "blue" not in fl, "red_ok": "red" not in fl, "swir1_ok": "swir1" not in fl, "n_bands": q.get("n_bands")})
allp = pd.DataFrame(rows); allp.to_csv(f"{PAIRS}/all_pairs_status.csv", index=False)
print(f"pairs found: {len(allp)} | {allp['status'].value_counts().to_dict()} | qc versions: {allp['qc_version'].value_counts().to_dict()}")

# ---- 2) PASS list minus EXCLUDE, with a completeness check ----
passp = allp[(allp["status"] == "PASS") & ~allp["pair"].isin(EXCLUDE)].copy(); ok_rows = []
for p in passp["pair"]:
    if all(glob.glob(f"{PAIRS}/{p}/{pat}") for pat in DELIVER): ok_rows.append(p)
    else: problems.append(f"pair {p}: PASS but a file is missing -> not copied")
passp = passp[passp["pair"].isin(ok_rows)]; keep = set(passp["pair"].astype(int))

# ---- 3) sync (stale copies -> _to_delete, nothing deleted) ----
def copy_if_changed(src, dst):
    if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(src) or os.path.getmtime(dst) < os.path.getmtime(src):
        shutil.copy2(src, dst); return True
    return False
moved = []
for d in glob.glob(f"{OUT}/*"):
    if os.path.isdir(d) and os.path.basename(d).isdigit() and int(os.path.basename(d)) not in keep:
        os.makedirs(f"{OUT}/_to_delete", exist_ok=True); dst = f"{OUT}/_to_delete/{os.path.basename(d)}"
        if os.path.exists(dst): dst += "_" + time.strftime("%Y%m%d_%H%M%S")
        shutil.move(d, dst); moved.append(int(os.path.basename(d)))
n_new = 0
for no in sorted(keep):
    src, dst = f"{PAIRS}/{no}", f"{OUT}/{no}"; os.makedirs(dst, exist_ok=True)
    files = sorted({os.path.basename(f) for pat in DELIVER for f in glob.glob(f"{src}/{pat}")})
    for f in files: n_new += copy_if_changed(f"{src}/{f}", f"{dst}/{f}")
    extra = [f for f in os.listdir(dst) if f not in files and f != "aux_layers.tif"]   # aux_layers.tif is made by FINAL_DATASET.ipynb A1
    if extra:                                                    # e.g. an older Landsat chip copied before EN-08b
        os.makedirs(f"{OUT}/_to_delete/{no}_extra", exist_ok=True)
        for f in extra: shutil.move(f"{dst}/{f}", f"{OUT}/_to_delete/{no}_extra/{f}")
        problems.append(f"pair {no}: moved old files to _to_delete: {extra}")
print(f"PASS pairs in PASSED-PAIRS: {len(keep)} ({n_new} files written or updated) | moved to _to_delete: {sorted(moved)}")

# ---- 4) CSV, band config, README ----
passp.to_csv(f"{OUT}/handover_pass.csv", index=False)
shutil.copy2(f"{ROOT}/config/enmap_bands.json", f"{OUT}/enmap_bands.json")
th = json.load(open(f"{PAIRS}/{sorted(keep)[0]}/pair_info.json"))["final_qc"]["thresholds"] if keep else {}
cmth, hz = th.get("cloud_mask", {}), th.get("haze", {})
nb, nr, ns = int((~passp["blue_ok"]).sum()), int((~passp["red_ok"]).sum()), int((~passp["swir1_ok"]).sum())
nmild = int(passp["haze_mild"].sum()); big = passp.loc[passp["large_shift"] == True, "pair"].tolist()
readme = f"""PASSED-PAIRS: EnMAP + Landsat 8/9 pairs that passed the quality rules (QC {QC_VERSION}, same rules as the Hyperion and EMIT pairs)
=========================================================================
Pairs: {len(passp)} in {passp['cell_id'].nunique()} cells (pair numbers as in ENMAP-LANDSAT/pairs/; gaps = not passed{', excluded: ' + str(EXCLUDE) if EXCLUDE else ''})
Climate groups (Koppen, Beck et al.): {passp['climate_group'].value_counts(dropna=False).to_dict()}

Each pair folder contains
  enmap_reflectance.tif  EnMAP L2A surface reflectance x 10000 (int16), nodata -32768, {passp['n_bands'].iloc[0] if len(passp) else ''} bands, 30 m,
                         wavelength and EnMAP band number in each band description
  landsat_<id>.tif       Landsat 8/9 on the SAME 30 m EnMAP grid (float32, NaN = no data, bilinear):
                         blue, green, red, nir, swir1, swir2 (surface reflectance), LST_K (kelvin), clear_frac (share of clear Landsat pixels)
  pair_info.json         scene IDs, dates, sun and viewing angles, AOD, EnMAP processor version, Koppen code / climate group, QC numbers
handover_pass.csv lists every pair with cell, dates, angles, climate and QC numbers. enmap_bands.json is the band configuration.

How the pairs were made
  - EnMAP L2A surface reflectance (DLR, processor version in pair_info), read for the 7.7 km cell only; reflectance = value x 0.0001
  - bands: 224 -> {passp['n_bands'].iloc[0] if len(passp) else ''} kept; dropped: oxygen 764 nm, water vapour 920-1004 and 1116-1152 nm, empty bands
    1343-1390 nm, the edges of the 1400 / 1900 nm gaps, 418-429 nm and 2415-2445 nm (noisy; data-driven, EN-05)
  - defective detector pixels (EnMAP defective_pixel_mask; not repaired by DLR) filled with the mean of good 3x3 (then 5x5) neighbours
    in the same band (test error about 5 %, equal to normal neighbour differences); share per pair: defective_filled_frac
  - EnMAP masks applied: cloud, cloud shadow, cirrus, snow. EnMAP's haze flag is recorded but NOT applied (it marked clear areas)
  - residual cloud and shadow masked by comparison with Landsat (5x5 median difference > {cmth.get('min_diff')} and factor {cmth.get('ratio_factor')},
    relative to the pair's own median ratio, buffered {cmth.get('dilation_px')} px)
  - Landsat 8/9 L2 resampled bilinearly onto the EnMAP 30 m grid; where the screening's Landsat scene ended inside the cell, the
    neighbouring scene (same overpass, or within +/-15 days) was used instead
  - EnMAP shifted by whole pixels only where the NDVI correlation with Landsat clearly improved (search +-{th.get('shift_search_px')} px)

PASS rules (design choices, not from the literature; identical for Hyperion, EMIT and EnMAP)
  joint coverage >= {th.get('min_cov_joint')}, NIR ratio within {th.get('ratio')}, NDVI r after alignment >= {th.get('min_ndvi_r')}, AOD <= {th.get('max_aod')},
  sun elevation >= {th.get('min_sun')} deg, Landsat clear >= {th.get('min_landsat_clear')}, no unresolved shift, large shifts confirmed,
  NDVI gain (mean EnMAP NDVI / mean Landsat NDVI over Landsat NDVI > {hz.get('veg_landsat_ndvi')}) >= {hz.get('min_ndvi_gain')}.
  FLAGS only: blue_ok / red_ok / swir1_ok (ratio outside 0.85-1.15): blue {nb}, red {nr}, SWIR1 {ns} of {len(passp)}; haze_mild: {nmild}.

Known caveats
  - EnMAP is a pointing satellite (off-nadir up to about 25 deg). The EnMAP/Landsat NIR ratio rises with the signed across-track
    angle (Spearman about 0.6): most likely directional reflectance (BRDF), not bad data. Angles are in pair_info for a later correction.
  - EnMAP SWIR reads about 15-20 % higher than Landsat SWIR; which sensor is closer to the truth is not established.
  - Several scenes share a cell: split training and validation BY cell_id, not by pair.
  - All EnMAP pairs are humid tropical cells (profile humid_coastal). Large shifts: {big if big else 'none'}.
  - PASS means "passed these checks", not perfect data.
"""
open(f"{OUT}/README.txt", "w").write(readme)

# ---- 5) summary ----
print("\nPASS by climate group:", passp["climate_group"].value_counts(dropna=False).to_dict())
print(f"cells: {passp['cell_id'].nunique()} | pairs per cell: {passp['cell_id'].value_counts().value_counts().sort_index().to_dict()}")
print(f"flags among delivered pairs: blue {nb}, red {nr}, swir1 {ns}, haze_mild {nmild} | Landsat replaced (EN-08b) among PASS: "
      f"{int(passp['pair'].isin([15, 19, 21, 23, 24, 33, 40]).sum())}")
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(OUT) if "_to_delete" not in dp for f in fs) / 1e9
print(f"PASSED-PAIRS size: {size:.2f} GB | location: {OUT}")
if problems: print("\nPROBLEMS:", *problems, sep="\n  ")

# Design choices, results and evidence
| Choice | Reason / evidence | Source |
|---|---|---|
| EnMAP **L2A** surface reflectance (land), processor 01.05.02, as delivered | Already atmospherically corrected and orthorectified by DLR; CARD4L surface-reflectance specification (STAC `card4l:*`) | DLR EnMAP L2A product specification (to cite after checking) |
| Read only the 0.07 deg cell from the Cloud-Optimised GeoTIFF on DLR's server | A full tile is about 470 MB; reading all 224 bands of the window in one call takes about 1 min per scene | Design decision (A2-A4 timings) |
| Keep EnMAP's own 30 m UTM grid; Landsat resampled onto it (bilinear) | Both are 30 m; Landsat's grid is offset by up to 15 m, bilinear avoids a half-pixel jump | Design decision |
| Drop 47 bands -> 177 | Oxygen and water-vapour absorption, empty bands, gap edges and noisy ends (roughness > 5 % in the pilot spectra, A5) | Design decision (data-driven) |
| Fill defective pixels from good 3x3 (then 5x5) neighbours | DLR did not repair them (30 % vs 5 % neighbour difference); 83 % of pixels would otherwise miss a band; fill error 5.2 % | Design decision (A4, A6) |
| Apply EnMAP cloud, cloud shadow, cirrus, snow masks; do not apply the haze flag | The cloud mask covers real clouds; the haze flag marked large clear areas (pairs 20: 32 %, 29: 39 %, 21) with NDVI r 0.89-0.96 | Design decision (figures) |
| Replace the screening's Landsat scene where it ends inside the cell (+-15 d) | 7 pairs had straight-edged "not clear" areas = scene edge; neighbouring scene of the same overpass in 5 cases | Design decision (B2) |
| AOD = `enmap:sceneAOT` / 1000 | Values 69-606, consistent with AOD 0.07-0.61 | My reading of the scale (to confirm in the product document) |
| QC v2.3 rules | Identical for Hyperion, EMIT and EnMAP | Design decision |

## Result
31 of 40 PASS in 23 cells (all tropical). Failures: NIR ratio 4 (pairs 2, 13, 25, 33), coverage 4 (9, 15, 17, 39: clouds or EnMAP tile edge),
haze 2 (25, 28: grey, washed-out EnMAP), NDVI r / unresolved shift 1 (17, very cloudy). No large shifts. Flags among PASS: blue 18,
red 10, SWIR1 16, haze_mild 1. Median EnMAP / Landsat ratios over PASS: blue 1.08, green 1.02, red 1.09, NIR 1.05, SWIR1 1.16, SWIR2 1.19.

## Findings to report
- **NIR depends on the viewing direction:** the EnMAP / Landsat NIR ratio correlates with the signed across-track off-nadir angle
  (Spearman 0.63, n = 40) but not with its size (0.09); all 4 NIR failures look from the same side (+7 to +15 deg). This is the
  signature of directional reflectance (BRDF), not of bad data, and probably also explains EMIT's "NIR too high" failures.
- **EnMAP SWIR reads 16-19 % higher than Landsat**; which sensor is closer to the truth is not established (flag only).
- Landsat was taken on average 40 minutes before EnMAP (median -0.03 d), except where the Landsat scene was replaced.

## Limits
- All EnMAP pairs are humid tropical cells (handover profile humid_coastal); 1,469 further EnMAP scenes exist in the screening.
- Several scenes share a cell: split training and validation by cell.
- The L2A atmospheric-correction method (I believe PACO) is not yet verified against the product document.

## References (check before citing)
- DLR. EnMAP HSI Level-2A product specification and the EOC Geoservice STAC collection `ENMAP_HSI_L2A`.
- Storch, T., et al. (2023). The EnMAP imaging spectroscopy mission towards operations. *Remote Sensing of Environment*.
- USGS Landsat Collection 2 Level-2 Science Product Guide (scale factors, QA_PIXEL).